In [1]:
pip install pdfplumber

   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 6.6/6.6 MB 38.6 MB/s  0:00:00
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   --------------- ------------------------ 1.3/3.5 MB 10.5 MB/s eta 0:00:01
   ---------------------------------------- 3.5/3.5 MB 21.3 MB/s  0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 3.7/3.7 MB 31.9 MB/s  0:00:00

   ---------------------------------------- 0/4 [pypdfium2]
   ---------------------------------------- 0/4 [pypdfium2]
   ---------------------------------------- 0/4 [pypdfium2]
   ---------- ----------------------------- 1/4 [cryptography]
   ---------- ----------------------------- 1/4 [cryptography]
   ---------- ----------------------------- 1/4 [cryptography]
   ---------- ----------------------------- 1/4 [cryptography]
   ---------- ----------------------------- 1/4 [cryptography

In [2]:
import pdfplumber
import pandas as pd
import os
import re

# Directory containing your downloaded state PDFs
pdf_dir = './state_pdfs'
all_state_data = []

def extract_state_data(pdf_path):
    data = {}
    with pdfplumber.open(pdf_path) as pdf:
        # Get State Name from filename or first page
        state_name = os.path.basename(pdf_path).split('-')[1].title()
        data['State'] = state_name
        
        # --- Page 4: Participation Rate ---
        page4 = pdf.pages[3]
        text4 = page4.extract_text()
        participation = re.search(r'SAT Participation Rate\s+(\d+%)', text4)
        data['SAT Participation Rate'] = participation.group(1) if participation else None
        
        # --- Page 5: Performance and Demographics ---
        page5 = pdf.pages[4]
        tables = page5.extract_tables()
        
        # Table 1: Total Performance (Total, ERW, Math)
        # Coordinates vary slightly, so we look for the 'Total' row
        perf_table = tables[0]
        for row in perf_table:
            if 'Total' in row:
                # Clean row to remove empty strings/None
                clean_row = [i for i in row if i]
                data['Mean Total'] = clean_row[2]
                data['Mean ERW'] = clean_row[3]
                data['Mean Math'] = clean_row[4]
                break

        # Table 2: Race/Ethnicity (Extracting common categories)
        race_table = tables[1]
        for row in race_table:
            if 'Asian' in row: data['% Asian'] = row[2]
            if 'Black/African American' in row: data['% Black'] = row[2]
            if 'Hispanic/Latino' in row: data['% Hispanic'] = row[2]
            if 'White' in row: data['% White'] = row[2]

        # Table 5: Parental Education
        edu_table = tables[4]
        for row in edu_table:
            if "Bachelor's Degree" in row: data['% Parents Bachelor'] = row[2]
            if "Graduate Degree" in row: data['% Parents Graduate'] = row[2]
            
    return data

# Process all files
for filename in os.listdir(pdf_dir):
    if filename.endswith(".pdf"):
        try:
            print(f"Processing {filename}...")
            state_record = extract_state_data(os.path.join(pdf_dir, filename))
            all_state_data.append(state_record)
        except Exception as e:
            print(f"Error processing {filename}: {e}")

# Create DataFrame
df = pd.DataFrame(all_state_data)

# Data Cleaning: Convert percentages to floats for your model
cols_to_fix = ['SAT Participation Rate', '% Asian', '% Black', '% Hispanic', 
               '% White', '% Parents Bachelor', '% Parents Graduate']

for col in cols_to_fix:
    df[col] = df[col].str.rstrip('%').astype('float') / 100.0

df.to_csv('sat_dataset_2025.csv', index=False)
print("Dataset successfully created: sat_dataset_2025.csv")

Processing 2025-alabama-sat-suite-of-assessments-annual-report ADA-v0.2.pdf...
Error processing 2025-alabama-sat-suite-of-assessments-annual-report ADA-v0.2.pdf: list index out of range
Processing 2025-alaska-sat-suite-of-assessments-annual-report ADA-v0.2.pdf...
Error processing 2025-alaska-sat-suite-of-assessments-annual-report ADA-v0.2.pdf: list index out of range
Processing 2025-arizona-sat-suite-of-assessments-annual-report ADA-v0.2.pdf...
Error processing 2025-arizona-sat-suite-of-assessments-annual-report ADA-v0.2.pdf: list index out of range
Processing 2025-arkansas-sat-suite-of-assessments-annual-report ADA-v0.2.pdf...
Error processing 2025-arkansas-sat-suite-of-assessments-annual-report ADA-v0.2.pdf: list index out of range
Processing 2025-california-sat-suite-of-assessments-annual-report ADA-v0.2.pdf...
Error processing 2025-california-sat-suite-of-assessments-annual-report ADA-v0.2.pdf: list index out of range
Processing 2025-colorado-sat-suite-of-assessments-annual-report 

KeyError: '% Asian'

In [3]:
import pdfplumber
import pandas as pd
import os
import re

pdf_dir = './state_pdfs'
all_state_data = []

def extract_state_data(pdf_path):
    # Define the structure with default None to prevent KeyErrors later
    data = {
        'State': os.path.basename(pdf_path).split('-')[1].title(),
        'SAT Participation Rate': None,
        'Mean Total': None, 'Mean ERW': None, 'Mean Math': None,
        '% Asian': None, '% Black': None, '% Hispanic': None, '% White': None,
        '% Parents Bachelor': None, '% Parents Graduate': None
    }
    
    with pdfplumber.open(pdf_path) as pdf:
        # --- Page 4: Participation ---
        page4_text = pdf.pages[3].extract_text()
        part_match = re.search(r'SAT Participation Rate\s+(\d+%)', page4_text)
        if part_match:
            data['SAT Participation Rate'] = part_match.group(1)

        # --- Page 5: Performance & Demographics ---
        page5 = pdf.pages[4]
        tables = page5.extract_tables()
        
        for table in tables:
            for row in table:
                # Clean row: remove Nones and newlines
                row = [str(cell).replace('\n', ' ').strip() for cell in row if cell]
                
                # Identify Performance Table
                if 'Total' in row and any(score in row for score in ['ERW', 'Math']):
                    # In DE report, Mean Total is usually at index 4 of the 'Total' row [cite: 223]
                    # Adjust indices based on the specific PDF structure found
                    data['Mean Total'] = row[4] if len(row) > 4 else None
                    data['Mean ERW'] = row[5] if len(row) > 5 else None
                    data['Mean Math'] = row[6] if len(row) > 6 else None

                # Identify Race/Ethnicity Table [cite: 224]
                if 'Asian' in row: data['% Asian'] = row[2] if len(row) > 2 else None
                if 'Black/African American' in row: data['% Black'] = row[2]
                if 'Hispanic/Latino' in row: data['% Hispanic'] = row[2]
                if 'White' in row: data['% White'] = row[2]

                # Identify Parental Education Table [cite: 227]
                if "Bachelor's Degree" in row: data['% Parents Bachelor'] = row[2]
                if "Graduate Degree" in row: data['% Parents Graduate'] = row[2]
            
    return data

# Process files
for filename in os.listdir(pdf_dir):
    if filename.endswith(".pdf"):
        try:
            all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))
        except Exception as e:
            print(f"Error in {filename}: {e}")

df = pd.DataFrame(all_state_data)

# Safe Percent Conversion: Only convert if the column exists and has data
cols_to_fix = ['SAT Participation Rate', '% Asian', '% Black', '% Hispanic', 
               '% White', '% Parents Bachelor', '% Parents Graduate']

for col in cols_to_fix:
    if col in df.columns:
        # Convert strings like "94%" to 0.94, handling None/NaN safely
        df[col] = df[col].apply(lambda x: float(x.rstrip('%')) / 100.0 if isinstance(x, str) and '%' in x else None)

df.to_csv('sat_dataset_2025.csv', index=False)

Error in 2025-alabama-sat-suite-of-assessments-annual-report ADA-v0.2.pdf: list index out of range
Error in 2025-alaska-sat-suite-of-assessments-annual-report ADA-v0.2.pdf: list index out of range
Error in 2025-arizona-sat-suite-of-assessments-annual-report ADA-v0.2.pdf: list index out of range
Error in 2025-arkansas-sat-suite-of-assessments-annual-report ADA-v0.2.pdf: list index out of range
Error in 2025-california-sat-suite-of-assessments-annual-report ADA-v0.2.pdf: list index out of range


In [5]:
import pdfplumber
import pandas as pd
import os
import re

pdf_dir = './state_pdfs'
all_state_data = []

def extract_state_data(pdf_path):
    # Initialize with default "Not Found" to see where extraction might be failing
    data = {
        'State': os.path.basename(pdf_path).split('-')[1].title() if '-' in pdf_path else "Unknown",
        'SAT Participation Rate': "0%",
        'Mean Total': 0, 'Mean ERW': 0, 'Mean Math': 0,
        '% Asian': "0%", '% Black': "0%", '% Hispanic': "0%", '% White': "0%",
        '% Parents Bachelor': "0%", '% Parents Graduate': "0%"
    }
    
    with pdfplumber.open(pdf_path) as pdf:
        # Page 4: Participation [cite: 201]
        text4 = pdf.pages[3].extract_text()
        part_match = re.search(r'SAT Participation Rate\s+(\d+%)', text4)
        if part_match: data['SAT Participation Rate'] = part_match.group(1)

        # Page 5: Performance & Demographics [cite: 225, 233]
        page5 = pdf.pages[4]
        tables = page5.extract_tables()
        
        for table in tables:
            for row in table:
                # Remove None and clean whitespace/newlines
                clean_row = [str(c).replace('\n', ' ').strip() for c in row if c is not None]
                
                # 1. Total Performance (Look for exactly 'Total' and 'Mean Score')
                # if 'Total' in clean_row and '933' in clean_row: # Using DE example score as anchor
                #      # Standard College Board layout: Total is often the first index, Mean is middle
                #      data['Mean Total'] = clean_row[4] if len(clean_row) > 4 else None
                #      data['Mean ERW'] = clean_row[5] if len(clean_row) > 5 else None
                #      data['Mean Math'] = clean_row[6] if len(clean_row) > 6 else None

                # CHANGE THIS PART:
                if 'Total' in clean_row and len(clean_row) > 6:    # Use the 'Number' of test takers as a secondary check if needed
                    data['Mean Total'] = clean_row[4]
                    data['Mean ERW'] = clean_row[5]
                    data['Mean Math'] = clean_row[6]

                # 2. Race/Ethnicity Table [cite: 233]
                if 'Asian' in clean_row: data['% Asian'] = clean_row[2]
                if 'Black/African American' in clean_row: data['% Black'] = clean_row[2]
                if 'Hispanic/Latino' in clean_row: data['% Hispanic'] = clean_row[2]
                if 'White' in clean_row: data['% White'] = clean_row[2]

                # 3. Parental Education Table [cite: 233]
                if "Bachelor's Degree" in clean_row: data['% Parents Bachelor'] = clean_row[2]
                if "Graduate Degree" in clean_row: data['% Parents Graduate'] = clean_row[2]
            
    return data

# Process files
for filename in os.listdir(pdf_dir):
    if filename.endswith(".pdf"):
        print(f"Reading: {filename}...")
        res = extract_state_data(os.path.join(pdf_dir, filename))
        all_state_data.append(res)

df = pd.DataFrame(all_state_data)

# Final formatting: convert all percentage strings to decimals
for col in df.columns:
    if '%' in str(df[col].iloc[0]):
        df[col] = df[col].str.rstrip('%').astype('float') / 100.0

df.to_csv('sat_final_dataset.csv', index=False)
print("Finished. Preview of data:")
print(df.head())

Reading: 2025-alabama-sat-suite-of-assessments-annual-report ADA-v0.2.pdf...


IndexError: list index out of range

In [6]:
import pdfplumber
import pandas as pd
import os
import re

pdf_dir = './state_pdfs'
all_state_data = []

def extract_state_data(pdf_path):
    data = {
        'State': os.path.basename(pdf_path).split('-')[1].title() if '-' in pdf_path else "Unknown",
        'SAT Participation Rate': "0%",
        'Mean Total': 0, 'Mean ERW': 0, 'Mean Math': 0,
        '% Asian': "0%", '% Black': "0%", '% Hispanic': "0%", '% White': "0%",
        '% Parents Bachelor': "0%", '% Parents Graduate': "0%"
    }
    
    with pdfplumber.open(pdf_path) as pdf:
        # [cite_start]Page 4: Participation [cite: 193]
        page4_text = pdf.pages[3].extract_text()
        part_match = re.search(r'SAT Participation Rate\s+(\d+%)', page4_text)
        if part_match:
            data['SAT Participation Rate'] = part_match.group(1)

        # [cite_start]Page 5: Performance & Demographics [cite: 219, 220, 224]
        page5 = pdf.pages[4]
        tables = page5.extract_tables()
        
        for table in tables:
            for row in table:
                # [cite_start]Collapse empty cells and remove newlines [cite: 220]
                clean_row = [str(c).replace('\n', ' ').strip() for c in row if c and str(c).strip()]
                
                if not clean_row:
                    continue

                # [cite_start]1. Total Performance - Dynamically find by 'Total' keyword and score length [cite: 220]
                if clean_row[0] == 'Total' and len(clean_row) >= 4:
                    # In standard reports, the last 3 numeric values are usually Total, ERW, Math
                    # [cite_start]For DE: [Total, 11,249, 933, 480, 453...] [cite: 220]
                    scores = [s for s in clean_row if s.isdigit() and len(s) >= 3]
                    if len(scores) >= 3:
                        data['Mean Total'] = scores[0]
                        data['Mean ERW'] = scores[1]
                        data['Mean Math'] = scores[2]

                # [cite_start]2. Race/Ethnicity Table - Check length before indexing [cite: 220]
                if len(clean_row) >= 3:
                    if 'Asian' in clean_row[0]: data['% Asian'] = clean_row[2]
                    if 'Black/African American' in clean_row[0]: data['% Black'] = clean_row[2]
                    if 'Hispanic/Latino' in clean_row[0]: data['% Hispanic'] = clean_row[2]
                    if 'White' in clean_row[0]: data['% White'] = clean_row[2]

                # [cite_start]3. Parental Education Table [cite: 224]
                if len(clean_row) >= 3:
                    if "Bachelor's Degree" in clean_row[0]: data['% Parents Bachelor'] = clean_row[2]
                    if "Graduate Degree" in clean_row[0]: data['% Parents Graduate'] = clean_row[2]
            
    return data

# Process files
for filename in os.listdir(pdf_dir):
    if filename.endswith(".pdf"):
        print(f"Reading: {filename}...")
        res = extract_state_data(os.path.join(pdf_dir, filename))
        all_state_data.append(res)

df = pd.DataFrame(all_state_data)

# Final formatting: convert all percentage strings to decimals
for col in df.columns:
    if '%' in str(df[col].iloc[0]):
        df[col] = df[col].str.rstrip('%').astype('float') / 100.0

df.to_csv('sat_final_dataset.csv', index=False)
print("Finished. Preview of data:")
print(df.head())

Reading: 2025-alabama-sat-suite-of-assessments-annual-report ADA-v0.2.pdf...
Reading: 2025-alaska-sat-suite-of-assessments-annual-report ADA-v0.2.pdf...
Reading: 2025-arizona-sat-suite-of-assessments-annual-report ADA-v0.2.pdf...
Reading: 2025-arkansas-sat-suite-of-assessments-annual-report ADA-v0.2.pdf...
Reading: 2025-california-sat-suite-of-assessments-annual-report ADA-v0.2.pdf...
Finished. Preview of data:
        State  SAT Participation Rate  Mean Total  Mean ERW  Mean Math  \
0     Alabama                    0.03           0         0          0   
1      Alaska                    0.27           0         0          0   
2     Arizona                    0.10           0         0          0   
3    Arkansas                    0.02           0         0          0   
4  California                    0.26           0         0          0   

   % Asian  % Black  % Hispanic  % White  % Parents Bachelor  \
0      0.0      0.0         0.0      0.0                 0.0   
1      0.0  

In [7]:
import pdfplumber
import pandas as pd
import os
import re

pdf_dir = './state_pdfs'
all_state_data = []

def get_percentage(row):
    """Helper to find the first cell in a list that looks like a percentage."""
    for cell in row:
        if '%' in str(cell):
            return str(cell).strip()
    return "0%"

def get_scores(row):
    """Helper to find SAT scores in a row. Mean Total is 400-1600, Sections are 200-800."""
    # Find all 3 or 4 digit numbers in the row
    nums = [int(s) for s in row if str(s).isdigit() and 200 <= int(s) <= 1600]
    # In CB reports, the order is usually [Total, ERW, Math]
    if len(nums) >= 3:
        # Total score should be the highest
        total = max(nums)
        # Remaining numbers are likely ERW and Math
        others = [n for n in nums if n != total]
        if len(others) >= 2:
            return total, others[0], others[1]
    return 0, 0, 0

def extract_state_data(pdf_path):
    # Initialize dictionary
    state_name = os.path.basename(pdf_path).replace('2025-', '').split('-')[0].title()
    data = {
        'State': state_name,
        'SAT Participation Rate': "0%",
        'Mean Total': 0, 'Mean ERW': 0, 'Mean Math': 0,
        '% Asian': "0%", '% Black': "0%", '% Hispanic': "0%", '% White': "0%",
        '% Parents Bachelor': "0%", '% Parents Graduate': "0%"
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # --- Page 4: Participation ---
            page4_text = pdf.pages[3].extract_text()
            part_match = re.search(r'SAT Participation Rate\s+(\d+%)', page4_text)
            if part_match:
                data['SAT Participation Rate'] = part_match.group(1)

            # --- Page 5: Loop through all tables and all rows ---
            for page_idx in [4]: # Checking Page 5
                page = pdf.pages[page_idx]
                tables = page.extract_tables()
                for table in tables:
                    for row in table:
                        # Clean the row: remove None, newlines, and empty strings
                        clean_row = [str(c).replace('\n', ' ').strip() for c in row if c and str(c).strip()]
                        if not clean_row: continue
                        
                        row_str = " ".join(clean_row).lower()

                        # 1. Capture Performance (Total Row)
                        if 'total' in clean_row[0].lower() and len(clean_row) > 3:
                            t, e, m = get_scores(clean_row)
                            if t > 0:
                                data['Mean Total'], data['Mean ERW'], data['Mean Math'] = t, e, m

                        # 2. Capture Race/Ethnicity
                        if 'asian' in row_str: data['% Asian'] = get_percentage(clean_row)
                        if 'black' in row_str: data['% Black'] = get_percentage(clean_row)
                        if 'hispanic' in row_str: data['% Hispanic'] = get_percentage(clean_row)
                        if 'white' in row_str: data['% White'] = get_percentage(clean_row)

                        # 3. Capture Parental Education
                        if "bachelor's" in row_str: data['% Parents Bachelor'] = get_percentage(clean_row)
                        if "graduate" in row_str: data['% Parents Graduate'] = get_percentage(clean_row)
    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
            
    return data

# Main Execution
for filename in os.listdir(pdf_dir):
    if filename.endswith(".pdf"):
        print(f"Processing: {filename}...")
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

# Create DataFrame and Clean formatting
df = pd.DataFrame(all_state_data)

# Convert all % columns to floats
cols_to_fix = [c for c in df.columns if '%' in c or 'Rate' in c]
for col in cols_to_fix:
    df[col] = df[col].str.rstrip('%').astype('float') / 100.0

df.to_csv('sat_final_dataset.csv', index=False)
print("\nExtraction Complete! Preview:")
print(df.head())

Processing: 2025-alabama-sat-suite-of-assessments-annual-report ADA-v0.2.pdf...
Processing: 2025-alaska-sat-suite-of-assessments-annual-report ADA-v0.2.pdf...
Processing: 2025-arizona-sat-suite-of-assessments-annual-report ADA-v0.2.pdf...
Processing: 2025-arkansas-sat-suite-of-assessments-annual-report ADA-v0.2.pdf...
Processing: 2025-california-sat-suite-of-assessments-annual-report ADA-v0.2.pdf...


ValueError: could not convert string to float: '234 15% 1297 632 665 86% 92% 90% 4'

In [9]:
import pdfplumber
import pandas as pd
import os
import re

# Directory containing your downloaded state PDFs
pdf_dir = './state_pdfs'
all_state_data = []

def safe_float_convert(value):
    """
    Extracts the first number or percentage from a messy string.
    Example: '234 15% 1297' -> 0.15
    """
    if pd.isna(value) or value == 0 or value == "0%":
        return 0.0
    
    val_str = str(value)
    
    # Check for percentage first (for participation/demographics)
    match = re.search(r'(\d+)%', val_str)
    if match:
        return float(match.group(1)) / 100.0
    
    # If no % sign, look for a stand-alone number (for SAT scores)
    nums = re.findall(r'\d+', val_str.replace(',', '')) # handle 1,249
    if nums:
        # Return the first significant number found
        return float(nums[0])
        
    return 0.0

def get_percentage(row):
    """Helper to find the first cell in a list that looks like a percentage."""
    for cell in row:
        if '%' in str(cell):
            return str(cell).strip()
    return "0%"

def get_scores(row):
    """Finds SAT scores by looking for numbers in valid SAT ranges."""
    # Convert row to numbers, ignoring non-numeric strings
    nums = []
    for s in row:
        clean_s = str(s).replace(',', '').strip()
        if clean_s.isdigit():
            n = int(clean_s)
            if 200 <= n <= 1600:
                nums.append(n)
    
    # Standard CB layout: Total score is usually the first/highest
    if len(nums) >= 3:
        total = nums[0]
        erw = nums[1]
        math = nums[2]
        return total, erw, math
    return 0, 0, 0

def extract_state_data(pdf_path):
    # Determine State Name from Filename
    file_name = os.path.basename(pdf_path)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    
    data = {
        'State': state_name,
        'SAT Participation Rate': "0%",
        'Mean Total': 0, 'Mean ERW': 0, 'Mean Math': 0,
        '% Asian': "0%", '% Black': "0%", '% Hispanic': "0%", '% White': "0%",
        '% Parents Bachelor': "0%", '% Parents Graduate': "0%"
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # --- Page 4: Participation ---
            page4_text = pdf.pages[3].extract_text()
            part_match = re.search(r'SAT Participation Rate\s+(\d+%)', page4_text)
            if part_match:
                data['SAT Participation Rate'] = part_match.group(1)

            # --- Page 5: Performance & Demographics ---
            page5 = pdf.pages[4]
            tables = page5.extract_tables()
            for table in tables:
                for row in table:
                    # Clean the row
                    clean_row = [str(c).replace('\n', ' ').strip() for c in row if c and str(c).strip()]
                    if not clean_row: continue
                    
                    row_str = " ".join(clean_row).lower()

                    # 1. Capture Performance (Total Row)
                    if 'total' in clean_row[0].lower():
                        t, e, m = get_scores(clean_row)
                        if t > 0:
                            data['Mean Total'], data['Mean ERW'], data['Mean Math'] = t, e, m

                    # 2. Capture Race/Ethnicity
                    if 'asian' in row_str: data['% Asian'] = get_percentage(clean_row)
                    if 'black' in row_str: data['% Black'] = get_percentage(clean_row)
                    if 'hispanic' in row_str: data['% Hispanic'] = get_percentage(clean_row)
                    if 'white' in row_str: data['% White'] = get_percentage(clean_row)

                    # 3. Capture Parental Education
                    if "bachelor's" in row_str: data['% Parents Bachelor'] = get_percentage(clean_row)
                    if "graduate" in row_str: data['% Parents Graduate'] = get_percentage(clean_row)
                    
    except Exception as e:
        print(f"Skipping {state_name} due to error: {e}")
            
    return data

# --- Main Execution ---
print("Starting extraction...")
for filename in os.listdir(pdf_dir):
    if filename.endswith(".pdf"):
        print(f"Processing: {filename}")
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

# Create DataFrame
df = pd.DataFrame(all_state_data)

# --- Safe Data Cleaning / Formatting ---
# Convert % columns
pct_cols = [c for c in df.columns if '%' in c or 'Rate' in c]
for col in pct_cols:
    df[col] = df[col].apply(safe_float_convert)

# Convert Score columns
score_cols = ['Mean Total', 'Mean ERW', 'Mean Math']
for col in score_cols:
    df[col] = df[col].apply(safe_float_convert)

# Save to CSV
df.to_csv('sat_final_dataset.csv', index=False)
print("\nSuccess! Dataset saved as 'sat_final_dataset.csv'")
print(df.head())

Starting extraction...
Processing: 2025-alabama-sat-suite-of-assessments-annual-report ADA-v0.2.pdf
Processing: 2025-alaska-sat-suite-of-assessments-annual-report ADA-v0.2.pdf
Processing: 2025-arizona-sat-suite-of-assessments-annual-report ADA-v0.2.pdf
Processing: 2025-arkansas-sat-suite-of-assessments-annual-report ADA-v0.2.pdf
Processing: 2025-california-sat-suite-of-assessments-annual-report ADA-v0.2.pdf

Success! Dataset saved as 'sat_final_dataset.csv'
        State  SAT Participation Rate  Mean Total  Mean ERW  Mean Math  \
0     Alabama                    0.03         0.0       0.0        0.0   
1      Alaska                    0.27         0.0       0.0        0.0   
2     Arizona                    0.10         0.0       0.0        0.0   
3    Arkansas                    0.02         0.0       0.0        0.0   
4  California                    0.26         0.0       0.0        0.0   

   % Asian  % Black  % Hispanic  % White  % Parents Bachelor  \
0     0.15      0.0        0.

In [10]:
import pdfplumber
import pandas as pd
import os
import re

pdf_dir = './state_pdfs'
all_state_data = []

def safe_float_convert(value):
    if pd.isna(value) or value == 0 or value == "0%":
        return 0.0
    val_str = str(value)
    match = re.search(r'(\d+)%', val_str)
    if match:
        return float(match.group(1)) / 100.0
    nums = re.findall(r'\d+', val_str.replace(',', ''))
    if nums:
        return float(nums[0])
    return 0.0

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    
    data = {
        'State': state_name,
        'SAT Participation Rate': "0%",
        'Mean Total': 0, 'Mean ERW': 0, 'Mean Math': 0,
        '% Asian': "0%", '% Black': "0%", '% Hispanic': "0%", '% White': "0%",
        '% Parents Bachelor': "0%", '% Parents Graduate': "0%"
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # --- Page 4: Participation ---
            page4_text = pdf.pages[3].extract_text()
            part_match = re.search(r'SAT Participation Rate\s+(\d+%)', page4_text)
            if part_match:
                data['SAT Participation Rate'] = part_match.group(1)

            # --- Page 5: Performance & Demographics ---
            page5 = pdf.pages[4]
            tables = page5.extract_tables()
            
            for table in tables:
                for row in table:
                    clean_row = [str(c).replace('\n', ' ').strip() for c in row if c and str(c).strip()]
                    if not clean_row: continue
                    row_str = " ".join(clean_row).lower()

                    # 1. FIXED PERFORMANCE EXTRACTION
                    # Look for rows containing 3-digit numbers that are likely SAT scores
                    # (Usually found in the 'Total' or 'SAT' rows)
                    potential_scores = [s for s in clean_row if s.isdigit() and 300 <= int(s) <= 1600]
                    if len(potential_scores) >= 3 and (data['Mean Total'] == 0):
                        # The first three are Total, ERW, Math
                        data['Mean Total'] = potential_scores[0]
                        data['Mean ERW'] = potential_scores[1]
                        data['Mean Math'] = potential_scores[2]

                    # 2. Capture Race/Ethnicity
                    if 'asian' in row_str: data['% Asian'] = next((c for c in clean_row if '%' in c), "0%")
                    if 'black' in row_str: data['% Black'] = next((c for c in clean_row if '%' in c), "0%")
                    if 'hispanic' in row_str: data['% Hispanic'] = next((c for c in clean_row if '%' in c), "0%")
                    if 'white' in row_str: data['% White'] = next((c for c in clean_row if '%' in c), "0%")

                    # 3. FIXED PARENTAL EDUCATION
                    # "Graduate Degree" and "Bachelor's Degree" are the specific labels
                    if "bachelor's" in row_str: 
                        data['% Parents Bachelor'] = next((c for c in clean_row if '%' in c), "0%")
                    if "graduate" in row_str: 
                        data['% Parents Graduate'] = next((c for c in clean_row if '%' in c), "0%")
                    
    except Exception as e:
        print(f"Skipping {state_name}: {e}")
            
    return data

# --- Execution ---
for filename in os.listdir(pdf_dir):
    if filename.endswith(".pdf"):
        print(f"Processing: {filename}")
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data)

# Convert all % columns
pct_cols = [c for c in df.columns if '%' in c or 'Rate' in c]
for col in pct_cols:
    df[col] = df[col].apply(safe_float_convert)

# Convert Score columns
score_cols = ['Mean Total', 'Mean ERW', 'Mean Math']
for col in score_cols:
    df[col] = df[col].apply(lambda x: int(x) if x else 0)

df.to_csv('sat_final_dataset.csv', index=False)
print("\nSuccess! Dataset saved. Preview below:")
print(df[['State', 'Mean Total', '% Parents Bachelor', '% Parents Graduate']].head())

Processing: 2025-alabama-sat-suite-of-assessments-annual-report ADA-v0.2.pdf
Processing: 2025-alaska-sat-suite-of-assessments-annual-report ADA-v0.2.pdf
Processing: 2025-arizona-sat-suite-of-assessments-annual-report ADA-v0.2.pdf
Processing: 2025-arkansas-sat-suite-of-assessments-annual-report ADA-v0.2.pdf
Processing: 2025-california-sat-suite-of-assessments-annual-report ADA-v0.2.pdf

Success! Dataset saved. Preview below:
        State  Mean Total  % Parents Bachelor  % Parents Graduate
0     Alabama           0                0.32                 0.0
1      Alaska           0                0.32                 0.0
2     Arizona           0                0.33                 0.0
3    Arkansas           0                0.32                 0.0
4  California           0                0.24                 0.0


In [14]:
import pdfplumber
import pandas as pd
import os
import re

# Directory containing your downloaded state PDFs
pdf_dir = './state_pdfs'
all_state_data = []

def robust_clean(value, is_percentage=True):
    """
    Extracts the first number or percentage from a messy string.
    Example: '32% 1150 580' -> 0.32
    Example: '1172 588 584' -> 1172.0
    """
    if not value or value == "0.0" or value == "0%":
        return 0.0
    
    val_str = str(value).replace(',', '')
    
    if is_percentage:
        # Look for a number followed by a percent sign
        match = re.search(r'(\d+)%', val_str)
        if match:
            return float(match.group(1)) / 100.0
    else:
        # Look for a 3 or 4 digit number (SAT scores)
        match = re.search(r'(\d{3,4})', val_str)
        if match:
            return float(match.group(1))
            
    return 0.0

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    # Extract state name from filename (assumes format 2025-statename-...)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    
    data = {
        'State': state_name,
        'SAT Participation Rate': 0.0,
        'Mean Total': 0.0, 'Mean ERW': 0.0, 'Mean Math': 0.0,
        '% Asian': 0.0, '% Black': 0.0, '% Hispanic': 0.0, '% White': 0.0,
        '% Parents Bachelor': 0.0, '% Parents Graduate': 0.0
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # Get text from relevant pages
            p4_text = pdf.pages[3].extract_text() if len(pdf.pages) > 3 else ""
            p5_text = pdf.pages[4].extract_text() if len(pdf.pages) > 4 else ""
            
            # --- 1. SCORES & PARTICIPATION (Regex Scan) ---
            # Participation (Page 4)
            part_match = re.search(r'SAT Participation Rate\s+(\d+%)', p4_text)
            if part_match:
                data['SAT Participation Rate'] = robust_clean(part_match.group(1))

            # Scores (Page 5) - Looks for: Total [Count] [TotalScore] [ERW] [Math]
            score_match = re.search(r'Total\s+[\d,]+\s+(\d{3,4})\s+(\d{2,3})\s+(\d{2,3})', p5_text)
            if score_match:
                data['Mean Total'] = float(score_match.group(1))
                data['Mean ERW'] = float(score_match.group(2))
                data['Mean Math'] = float(score_match.group(3))

            # --- 2. DEMOGRAPHICS & EDUCATION (Table Scan) ---
            if len(pdf.pages) > 4:
                tables = pdf.pages[4].extract_tables()
                for table in tables:
                    for row in table:
                        # Join row cells to search for keywords
                        row_cells = [str(c).strip() for c in row if c]
                        row_str = " ".join(row_cells).lower()
                        
                        # Match Demographics
                        if 'asian' in row_str: data['% Asian'] = robust_clean(row_str)
                        if 'black' in row_str: data['% Black'] = robust_clean(row_str)
                        if 'hispanic' in row_str: data['% Hispanic'] = robust_clean(row_str)
                        if 'white' in row_str: data['% White'] = robust_clean(row_str)
                        
                        # Match Education
                        if 'bachelor' in row_str: data['% Parents Bachelor'] = robust_clean(row_str)
                        if 'grad' in row_str or 'prof' in row_str: 
                            data['% Parents Graduate'] = robust_clean(row_str)

            # --- 3. FALLBACK (Regex Scan for Education) ---
            # If the table structure was broken, search the raw text for specific phrases
            if data['% Parents Bachelor'] == 0:
                bach_match = re.search(r"Bachelor's Degree\s+[\d,]+\s+(\d+%)", p5_text)
                if bach_match: data['% Parents Bachelor'] = robust_clean(bach_match.group(1))
            
            if data['% Parents Graduate'] == 0:
                grad_match = re.search(r"Graduate Degree\s+[\d,]+\s+(\d+%)", p5_text)
                if grad_match: data['% Parents Graduate'] = robust_clean(grad_match.group(1))
                    
    except Exception as e:
        print(f"Skipping {state_name} due to unexpected error: {e}")
            
    return data

# --- Main Logic ---
print(f"Scanning directory: {pdf_dir}...")
for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        print(f"Processing: {filename}")
        result = extract_state_data(os.path.join(pdf_dir, filename))
        all_state_data.append(result)

# Create DataFrame
df = pd.DataFrame(all_state_data)

# Export to CSV
output_file = 'sat_comprehensive_dataset_2025.csv'
df.to_csv(output_file, index=False)

print(f"\nSuccessfully generated {output_file}")
print("Preview of the extracted data:")
print(df[['State', 'Mean Total', 'SAT Participation Rate', '% Parents Bachelor', '% Parents Graduate']].head())

Scanning directory: ./state_pdfs...
Processing: 2025-alabama-sat-suite-of-assessments-annual-report ADA-v0.2.pdf
Processing: 2025-alaska-sat-suite-of-assessments-annual-report ADA-v0.2.pdf
Processing: 2025-arizona-sat-suite-of-assessments-annual-report ADA-v0.2.pdf
Processing: 2025-arkansas-sat-suite-of-assessments-annual-report ADA-v0.2.pdf
Processing: 2025-california-sat-suite-of-assessments-annual-report ADA-v0.2.pdf

Successfully generated sat_comprehensive_dataset_2025.csv
Preview of the extracted data:
        State  Mean Total  SAT Participation Rate  % Parents Bachelor  \
0     Alabama      1172.0                    0.03                0.32   
1      Alaska      1097.0                    0.27                0.32   
2     Arizona      1194.0                    0.10                0.33   
3    Arkansas      1177.0                    0.02                0.32   
4  California      1096.0                    0.26                0.24   

   % Parents Graduate  
0                0.38  

In [15]:
import pdfplumber
import pandas as pd
import os
import re

# Directory containing your downloaded state PDFs
pdf_dir = './state_pdfs'
all_state_data = []

def robust_clean(value, is_percentage=True):
    if not value or value == "0.0" or value == "0%":
        return 0.0
    val_str = str(value).replace(',', '')
    if is_percentage:
        match = re.search(r'(\d+)%', val_str)
        if match:
            return float(match.group(1)) / 100.0
    else:
        match = re.search(r'(\d{3,4})', val_str)
        if match:
            return float(match.group(1))
    return 0.0

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    
    data = {
        'State': state_name,
        'SAT Participation Rate': 0.0,
        'Mean Total': 0.0, 'Mean ERW': 0.0, 'Mean Math': 0.0,
        '% Asian': 0.0, '% Black': 0.0, '% Hispanic': 0.0, '% White': 0.0,
        '% Parents Bachelor': 0.0, '% Parents Graduate': 0.0
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            p4_text = pdf.pages[3].extract_text() if len(pdf.pages) > 3 else ""
            p5_text = pdf.pages[4].extract_text() if len(pdf.pages) > 4 else ""
            
            # --- 1. SCORES & PARTICIPATION ---
            part_match = re.search(r'SAT Participation Rate\s+(\d+%)', p4_text)
            if part_match:
                data['SAT Participation Rate'] = robust_clean(part_match.group(1))

            score_match = re.search(r'Total\s+[\d,]+\s+(\d{3,4})\s+(\d{2,3})\s+(\d{2,3})', p5_text)
            if score_match:
                data['Mean Total'] = float(score_match.group(1))
                data['Mean ERW'] = float(score_match.group(2))
                data['Mean Math'] = float(score_match.group(3))

            # --- 2. TABLE SCAN (Demographics & Education) ---
            tables = pdf.pages[4].extract_tables()
            for table in tables:
                for row in table:
                    row_cells = [str(c).strip() for c in row if c]
                    row_str = " ".join(row_cells).lower()
                    
                    # Regex-based Keyword Matching to handle slashes (e.g. Black/African American)
                    if 'asian' in row_str: data['% Asian'] = robust_clean(row_str)
                    if 'black' in row_str or 'african american' in row_str: 
                        data['% Black'] = robust_clean(row_str)
                    if 'hispanic' in row_str or 'latino' in row_str: 
                        data['% Hispanic'] = robust_clean(row_str)
                    if 'white' in row_str: data['% White'] = robust_clean(row_str)
                    
                    if 'bachelor' in row_str: data['% Parents Bachelor'] = robust_clean(row_str)
                    if 'grad' in row_str or 'prof' in row_str: 
                        data['% Parents Graduate'] = robust_clean(row_str)

            # --- 3. FALLBACK (Regex for missing values) ---
            # Fallback specifically for Black/African American
            if data['% Black'] == 0:
                black_match = re.search(r"Black/African American\s+[\d,]+\s+(\d+%)", p5_text)
                if black_match: data['% Black'] = robust_clean(black_match.group(1))
                    
    except Exception as e:
        print(f"Skipping {state_name}: {e}")
            
    return data

# --- Execution ---
print("Extracting full dataset...")
for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data)
df.to_csv('sat_final_dataset.csv', index=False)

print("\nSuccess! Final Dataset Sample:")
print(df[['State', 'Mean Total', '% Black', '% Parents Graduate']].head())

Extracting full dataset...

Success! Final Dataset Sample:
        State  Mean Total  % Black  % Parents Graduate
0     Alabama      1172.0     0.18                 0.0
1      Alaska      1097.0     0.02                 0.0
2     Arizona      1194.0     0.04                 0.0
3    Arkansas      1177.0     0.08                 0.0
4  California      1096.0     0.04                 0.0


In [11]:
import pdfplumber
import pandas as pd
import os
import re

pdf_dir = './state_pdfs'
all_state_data = []

def safe_float_convert(value):
    if pd.isna(value) or value == 0 or value == "0%":
        return 0.0
    val_str = str(value)
    match = re.search(r'(\d+)%', val_str)
    if match:
        return float(match.group(1)) / 100.0
    nums = re.findall(r'\d+', val_str.replace(',', ''))
    return float(nums[0]) if nums else 0.0

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    
    data = {
        'State': state_name,
        'SAT Participation Rate': "0%",
        'Mean Total': 0, 'Mean ERW': 0, 'Mean Math': 0,
        '% Asian': "0%", '% Black': "0%", '% Hispanic': "0%", '% White': "0%",
        '% Parents Bachelor': "0%", '% Parents Graduate': "0%"
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # --- Page 4: Participation ---
            page4_text = pdf.pages[3].extract_text()
            part_match = re.search(r'SAT Participation Rate\s+(\d+%)', page4_text)
            if part_match:
                data['SAT Participation Rate'] = part_match.group(1)

            # --- Page 5: Full Page Scan ---
            page5 = pdf.pages[4]
            tables = page5.extract_tables()
            
            for table in tables:
                for row in table:
                    # Clean the row thoroughly
                    clean_row = [str(c).replace('\n', ' ').strip() for c in row if c and str(c).strip()]
                    if not clean_row: continue
                    row_str = " ".join(clean_row).lower()

                    # 1. Performance: Look for any row containing a score-like pattern
                    # Usually: [Label, Count, Total, ERW, Math] or [Label, Total, ERW, Math]
                    nums = [n for n in clean_row if n.replace(',', '').isdigit()]
                    if len(nums) >= 3:
                        # Identify scores: Total is 400-1600, Section is 200-800
                        # We look for the first occurrence of this set
                        scores = [int(n.replace(',', '')) for n in nums if 200 <= int(n.replace(',', '')) <= 1600]
                        if len(scores) >= 3 and data['Mean Total'] == 0:
                            # In CB reports, Total is usually the first of the three
                            data['Mean Total'] = scores[0]
                            data['Mean ERW'] = scores[1]
                            data['Mean Math'] = scores[2]

                    # 2. Race/Ethnicity
                    if 'asian' in row_str: data['% Asian'] = next((c for c in clean_row if '%' in c), "0%")
                    if 'black' in row_str: data['% Black'] = next((c for c in clean_row if '%' in c), "0%")
                    if 'hispanic' in row_str: data['% Hispanic'] = next((c for c in clean_row if '%' in c), "0%")
                    if 'white' in row_str: data['% White'] = next((c for c in clean_row if '%' in c), "0%")

                    # 3. Parental Education
                    if "bachelor" in row_str: 
                        data['% Parents Bachelor'] = next((c for c in clean_row if '%' in c), "0%")
                    if "graduate" in row_str: 
                        data['% Parents Graduate'] = next((c for c in clean_row if '%' in c), "0%")
                    
    except Exception as e:
        print(f"Error in {state_name}: {e}")
            
    return data

# --- Main Logic ---
print("Extracting data from PDFs...")
for filename in os.listdir(pdf_dir):
    if filename.endswith(".pdf"):
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data)

# Clean Percentages
pct_cols = [c for c in df.columns if '%' in c or 'Rate' in c]
for col in pct_cols:
    df[col] = df[col].apply(safe_float_convert)

# Ensure scores are numeric
score_cols = ['Mean Total', 'Mean ERW', 'Mean Math']
for col in score_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

df.to_csv('sat_final_dataset.csv', index=False)
print("\nSuccess! Dataset created.")
print(df[['State', 'Mean Total', '% Parents Bachelor', '% Parents Graduate']].head())

Extracting data from PDFs...

Success! Dataset created.
        State  Mean Total  % Parents Bachelor  % Parents Graduate
0     Alabama           0                0.32                 0.0
1      Alaska           0                0.32                 0.0
2     Arizona           0                0.33                 0.0
3    Arkansas           0                0.32                 0.0
4  California           0                0.24                 0.0


In [12]:
import pdfplumber
import pandas as pd
import os
import re

pdf_dir = './state_pdfs'
all_state_data = []

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    
    data = {
        'State': state_name,
        'SAT Participation Rate': "0%",
        'Mean Total': 0, 'Mean ERW': 0, 'Mean Math': 0,
        '% Asian': "0%", '% Black': "0%", '% Hispanic': "0%", '% White': "0%",
        '% Parents Bachelor': "0%", '% Parents Graduate': "0%"
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # --- Page 4: Participation ---
            page4_text = pdf.pages[3].extract_text()
            part_match = re.search(r'SAT Participation Rate\s+(\d+%)', page4_text)
            if part_match:
                data['SAT Participation Rate'] = part_match.group(1)

            # --- Page 5: Raw Text Scan (More reliable for Mean Total) ---
            page5 = pdf.pages[4]
            full_text = page5.extract_text()
            
            # Pattern for Mean Total: Usually follows "Total" or "SAT" and is 3-4 digits
            # We look for the first 3-digit number after "Total" in the performance section
            perf_match = re.search(r'Total\s+[\d,]+\s+(\d{3,4})\s+(\d{2,3})\s+(\d{2,3})', full_text)
            if perf_match:
                data['Mean Total'] = perf_match.group(1)
                data['Mean ERW'] = perf_match.group(2)
                data['Mean Math'] = perf_match.group(3)

            # --- Page 5: Table Scan (Demographics & Education) ---
            tables = page5.extract_tables()
            for table in tables:
                for row in table:
                    clean_row = [str(c).replace('\n', ' ').strip() for c in row if c and str(c).strip()]
                    if not clean_row: continue
                    row_str = " ".join(clean_row).lower()

                    # Race/Ethnicity (Working)
                    if 'asian' in row_str: data['% Asian'] = next((c for c in clean_row if '%' in c), "0%")
                    if 'black' in row_str: data['% Black'] = next((c for c in clean_row if '%' in c), "0%")
                    if 'hispanic' in row_str: data['% Hispanic'] = next((c for c in clean_row if '%' in c), "0%")
                    if 'white' in row_str: data['% White'] = next((c for c in clean_row if '%' in c), "0%")

                    # Parental Education (Refined matching)
                    if "bachelor" in row_str: 
                        data['% Parents Bachelor'] = next((c for c in clean_row if '%' in c), "0%")
                    # Catch 'graduate' or 'professional' degree
                    if "grad" in row_str or "prof" in row_str: 
                        # Grab the first percentage in a row that mentions graduation
                        for cell in clean_row:
                            if '%' in cell:
                                data['% Parents Graduate'] = cell
                                break
                    
    except Exception as e:
        print(f"Error in {state_name}: {e}")
            
    return data

# --- Cleaning and Saving ---
def safe_convert(val):
    if not val: return 0.0
    s = str(val).replace('%', '').strip()
    try:
        return float(s) / 100.0 if '%' in str(val) else float(s)
    except:
        return 0.0

print("Starting deep scan...")
for filename in os.listdir(pdf_dir):
    if filename.endswith(".pdf"):
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data)

# Apply cleaning
for col in df.columns:
    if col != 'State':
        df[col] = df[col].apply(safe_convert)

df.to_csv('sat_final_dataset.csv', index=False)
print("Done! Preview:")
print(df[['State', 'Mean Total', '% Parents Bachelor', '% Parents Graduate']].head())

Starting deep scan...
Done! Preview:
        State  Mean Total  % Parents Bachelor  % Parents Graduate
0     Alabama      1172.0                 0.0                 0.0
1      Alaska      1097.0                 0.0                 0.0
2     Arizona      1194.0                 0.0                 0.0
3    Arkansas      1177.0                 0.0                 0.0
4  California      1096.0                 0.0                 0.0


In [13]:
import pdfplumber
import pandas as pd
import os
import re

pdf_dir = './state_pdfs'
all_state_data = []

def robust_clean(value, is_percentage=True):
    """
    Extracts the first number or percentage from a potentially messy string.
    Example: '32% 1150 580' -> 0.32
    Example: '1172 588 584' -> 1172.0
    """
    if not value or value == "0%":
        return 0.0
    
    val_str = str(value)
    
    if is_percentage:
        # Look for a number followed by a percent sign
        match = re.search(r'(\d+)%', val_str)
        if match:
            return float(match.group(1)) / 100.0
    else:
        # Look for a 3 or 4 digit number (SAT scores)
        match = re.search(r'(\d{3,4})', val_str.replace(',', ''))
        if match:
            return float(match.group(1))
            
    return 0.0

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    
    data = {
        'State': state_name,
        'SAT Participation Rate': 0.0,
        'Mean Total': 0.0, 'Mean ERW': 0.0, 'Mean Math': 0.0,
        '% Asian': 0.0, '% Black': 0.0, '% Hispanic': 0.0, '% White': 0.0,
        '% Parents Bachelor': 0.0, '% Parents Graduate': 0.0
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # --- 1. FULL TEXT SCAN (Most reliable for Scores and Participation) ---
            p4_text = pdf.pages[3].extract_text()
            p5_text = pdf.pages[4].extract_text()
            
            # Extract Participation (Page 4)
            part_match = re.search(r'SAT Participation Rate\s+(\d+%)', p4_text)
            if part_match:
                data['SAT Participation Rate'] = robust_clean(part_match.group(1))

            # Extract Scores (Page 5)
            # Pattern: Total [Number of Takers] [Total Score] [ERW] [Math]
            score_match = re.search(r'Total\s+[\d,]+\s+(\d{3,4})\s+(\d{2,3})\s+(\d{2,3})', p5_text)
            if score_match:
                data['Mean Total'] = float(score_match.group(1))
                data['Mean ERW'] = float(score_match.group(2))
                data['Mean Math'] = float(score_match.group(3))

            # --- 2. TABLE SCAN (Demographics and Education) ---
            tables = pdf.pages[4].extract_tables()
            for table in tables:
                for row in table:
                    clean_row = [str(c).strip() for c in row if c]
                    row_str = " ".join(clean_row).lower()
                    
                    # Match Demographic Percentages
                    if 'asian' in row_str: data['% Asian'] = robust_clean(row_str)
                    if 'black' in row_str: data['% Black'] = robust_clean(row_str)
                    if 'hispanic' in row_str: data['% Hispanic'] = robust_clean(row_str)
                    if 'white' in row_str: data['% White'] = robust_clean(row_str)
                    
                    # Match Education Percentages
                    if 'bachelor' in row_str: data['% Parents Bachelor'] = robust_clean(row_str)
                    if 'grad' in row_str or 'prof' in row_str: 
                        data['% Parents Graduate'] = robust_clean(row_str)

            # --- 3. FALLBACK: TEXT SCAN FOR EDUCATION (If tables are too messy) ---
            if data['% Parents Bachelor'] == 0:
                bach_match = re.search(r"Bachelor's Degree\s+[\d,]+\s+(\d+%)", p5_text)
                if bach_match: data['% Parents Bachelor'] = robust_clean(bach_match.group(1))
            
            if data['% Parents Graduate'] == 0:
                grad_match = re.search(r"Graduate Degree\s+[\d,]+\s+(\d+%)", p5_text)
                if grad_match: data['% Parents Graduate'] = robust_clean(grad_match.group(1))
                    
    except Exception as e:
        print(f"Skipping {state_name} due to error: {e}")
            
    return data

# --- Main Execution ---
print("Scanning PDFs for Scores and Education...")
for filename in os.listdir(pdf_dir):
    if filename.endswith(".pdf"):
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data)

# Final Save
df.to_csv('sat_final_dataset.csv', index=False)
print("\nSuccess! Preview of Final Dataset:")
print(df[['State', 'Mean Total', 'SAT Participation Rate', '% Parents Bachelor', '% Parents Graduate']].head())

Scanning PDFs for Scores and Education...

Success! Preview of Final Dataset:
        State  Mean Total  SAT Participation Rate  % Parents Bachelor  \
0     Alabama      1172.0                    0.03                0.32   
1      Alaska      1097.0                    0.27                0.32   
2     Arizona      1194.0                    0.10                0.33   
3    Arkansas      1177.0                    0.02                0.32   
4  California      1096.0                    0.26                0.24   

   % Parents Graduate  
0                0.38  
1                0.25  
2                0.36  
3                0.36  
4                0.26  


In [16]:
import pdfplumber
import pandas as pd
import os
import re

# Directory containing your downloaded state PDFs
pdf_dir = './state_pdfs'
all_state_data = []

def robust_clean(value, is_percentage=True):
    if not value or value == "0.0" or value == "0%":
        return 0.0
    val_str = str(value).replace(',', '')
    if is_percentage:
        match = re.search(r'(\d+)%', val_str)
        if match:
            return float(match.group(1)) / 100.0
    else:
        match = re.search(r'(\d{3,4})', val_str)
        if match:
            return float(match.group(1))
    return 0.0

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    
    data = {
        'State': state_name,
        'SAT Participation Rate': 0.0,
        'Mean Total': 0.0, 'Mean ERW': 0.0, 'Mean Math': 0.0,
        '% Asian': 0.0, '% Black': 0.0, '% Hispanic': 0.0, '% White': 0.0,
        '% Parents Bachelor': 0.0, '% Parents Graduate': 0.0
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            p4_text = pdf.pages[3].extract_text() if len(pdf.pages) > 3 else ""
            p5_text = pdf.pages[4].extract_text() if len(pdf.pages) > 4 else ""
            
            # --- 1. SCORES & PARTICIPATION ---
            part_match = re.search(r'SAT Participation Rate\s+(\d+%)', p4_text)
            if part_match:
                data['SAT Participation Rate'] = robust_clean(part_match.group(1))

            score_match = re.search(r'Total\s+[\d,]+\s+(\d{3,4})\s+(\d{2,3})\s+(\d{2,3})', p5_text)
            if score_match:
                data['Mean Total'] = float(score_match.group(1))
                data['Mean ERW'] = float(score_match.group(2))
                data['Mean Math'] = float(score_match.group(3))

            # --- 2. TABLE SCAN (Demographics & Education) ---
            tables = pdf.pages[4].extract_tables()
            for table in tables:
                for row in table:
                    row_cells = [str(c).strip() for c in row if c]
                    row_str = " ".join(row_cells).lower()
                    
                    # Regex-based Keyword Matching to handle slashes (e.g. Black/African American)
                    if 'asian' in row_str: data['% Asian'] = robust_clean(row_str)
                    if 'black' in row_str or 'african american' in row_str: 
                        data['% Black'] = robust_clean(row_str)
                    if 'hispanic' in row_str or 'latino' in row_str: 
                        data['% Hispanic'] = robust_clean(row_str)
                    if 'white' in row_str: data['% White'] = robust_clean(row_str)
                    
                    if 'bachelor' in row_str: data['% Parents Bachelor'] = robust_clean(row_str)
                    if 'grad' in row_str or 'prof' in row_str: 
                        data['% Parents Graduate'] = robust_clean(row_str)

            # --- 3. FALLBACK (Regex for missing values) ---
            # Fallback specifically for Black/African American
            if data['% Black'] == 0:
                black_match = re.search(r"Black/African American\s+[\d,]+\s+(\d+%)", p5_text)
                if black_match: data['% Black'] = robust_clean(black_match.group(1))
                    
    except Exception as e:
        print(f"Skipping {state_name}: {e}")
            
    return data

# --- Execution ---
print("Extracting full dataset...")
for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data)
df.to_csv('sat_final_dataset.csv', index=False)

print("\nSuccess! Final Dataset Sample:")
print(df[['State', 'Mean Total', '% Black', '% Parents Graduate']].head())

Extracting full dataset...

Success! Final Dataset Sample:
        State  Mean Total  % Black  % Parents Graduate
0     Alabama      1172.0     0.18                 0.0
1      Alaska      1097.0     0.02                 0.0
2     Arizona      1194.0     0.04                 0.0
3    Arkansas      1177.0     0.08                 0.0
4  California      1096.0     0.04                 0.0


In [17]:
import pdfplumber
import pandas as pd
import os
import re

# Directory containing your downloaded state PDFs
pdf_dir = './state_pdfs'
all_state_data = []

def robust_clean(value, is_percentage=True):
    """
    Extracts the first number or percentage from a messy string.
    Example: 'Black/African American 1,249 15% 900' -> 0.15
    Example: 'Total 11,249 933 480 453' -> 933.0
    """
    if not value or value == "0.0" or value == "0%":
        return 0.0
    
    # Remove commas from large numbers (e.g., 1,249 -> 1249)
    val_str = str(value).replace(',', '')
    
    if is_percentage:
        # Search for a number followed by a percent sign
        match = re.search(r'(\d+)%', val_str)
        if match:
            return float(match.group(1)) / 100.0
    else:
        # Search for a 3 or 4 digit number (Standard SAT scores)
        match = re.search(r'(\d{3,4})', val_str)
        if match:
            return float(match.group(1))
            
    return 0.0

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    # Assumes filename format: '2025-statename-annual-report...'
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    
    data = {
        'State': state_name,
        'SAT Participation Rate': 0.0,
        'Mean Total': 0.0, 'Mean ERW': 0.0, 'Mean Math': 0.0,
        '% Asian': 0.0, '% Black': 0.0, '% Hispanic': 0.0, '% White': 0.0,
        '% Parents Bachelor': 0.0, '% Parents Graduate': 0.0
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # Extract raw text from key pages for Regex scanning
            p4_text = pdf.pages[3].extract_text() if len(pdf.pages) > 3 else ""
            p5_text = pdf.pages[4].extract_text() if len(pdf.pages) > 4 else ""
            
            # --- 1. PARTICIPATION & SCORES (Using Regex on raw text) ---
            
            # Participation Rate (usually on page 4)
            part_match = re.search(r'SAT Participation Rate\s+(\d+%)', p4_text)
            if part_match:
                data['SAT Participation Rate'] = robust_clean(part_match.group(1))

            # Mean Scores (Page 5: Total, ERW, Math)
            # Pattern: Total [Takers Count] [Mean Total] [Mean ERW] [Mean Math]
            score_match = re.search(r'Total\s+[\d,]+\s+(\d{3,4})\s+(\d{2,3})\s+(\d{2,3})', p5_text)
            if score_match:
                data['Mean Total'] = float(score_match.group(1))
                data['Mean ERW'] = float(score_match.group(2))
                data['Mean Math'] = float(score_match.group(3))

            # --- 2. DEMOGRAPHICS & EDUCATION (Using Table Grids) ---
            
            if len(pdf.pages) > 4:
                tables = pdf.pages[4].extract_tables()
                for table in tables:
                    for row in table:
                        # Join all cells in the row into a single string for keyword searching
                        row_cells = [str(c).strip() for c in row if c]
                        row_str = " ".join(row_cells).lower()
                        
                        # Flexible Demographic Matching (handles slashes and multi-line text)
                        if 'asian' in row_str: 
                            data['% Asian'] = robust_clean(row_str)
                        if 'black' in row_str or 'african american' in row_str: 
                            data['% Black'] = robust_clean(row_str)
                        if 'hispanic' in row_str or 'latino' in row_str: 
                            data['% Hispanic'] = robust_clean(row_str)
                        if 'white' in row_str: 
                            data['% White'] = robust_clean(row_str)
                        
                        # Parental Education Matching
                        if 'bachelor' in row_str: 
                            data['% Parents Bachelor'] = robust_clean(row_str)
                        if 'grad' in row_str or 'prof' in row_str: 
                            data['% Parents Graduate'] = robust_clean(row_str)

            # --- 3. LAST-RESORT FALLBACK (Regex on missing items) ---
            
            # Specifically check if Black demographic is still 0
            if data['% Black'] == 0:
                black_fallback = re.search(r"Black/African American\s+[\d,]+\s+(\d+%)", p5_text)
                if black_fallback: data['% Black'] = robust_clean(black_fallback.group(1))
            
            # Specifically check if Education is still 0
            if data['% Parents Bachelor'] == 0:
                bach_fallback = re.search(r"Bachelor's Degree\s+[\d,]+\s+(\d+%)", p5_text)
                if bach_fallback: data['% Parents Bachelor'] = robust_clean(bach_fallback.group(1))
                
            if data['% Parents Graduate'] == 0:
                grad_fallback = re.search(r"Graduate Degree\s+[\d,]+\s+(\d+%)", p5_text)
                if grad_fallback: data['% Parents Graduate'] = robust_clean(grad_fallback.group(1))
                    
    except Exception as e:
        print(f"Error processing {state_name}: {e}")
            
    return data

# --- Main Logic ---
print("Starting comprehensive extraction...")
for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        print(f"Processing: {filename}")
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

# Create DataFrame
df = pd.DataFrame(all_state_data)

# Export results
output_name = 'sat_final_master_dataset.csv'
df.to_csv(output_name, index=False)

print(f"\nExtraction complete. File saved as: {output_name}")
print("Final Data Preview:")
print(df.head())

Starting comprehensive extraction...
Processing: 2025-alabama-sat-suite-of-assessments-annual-report ADA-v0.2.pdf
Processing: 2025-alaska-sat-suite-of-assessments-annual-report ADA-v0.2.pdf
Processing: 2025-arizona-sat-suite-of-assessments-annual-report ADA-v0.2.pdf
Processing: 2025-arkansas-sat-suite-of-assessments-annual-report ADA-v0.2.pdf
Processing: 2025-california-sat-suite-of-assessments-annual-report ADA-v0.2.pdf

Extraction complete. File saved as: sat_final_master_dataset.csv
Final Data Preview:
        State  SAT Participation Rate  Mean Total  Mean ERW  Mean Math  \
0     Alabama                    0.03      1172.0     601.0      570.0   
1      Alaska                    0.27      1097.0     567.0      530.0   
2     Arizona                    0.10      1194.0     606.0      588.0   
3    Arkansas                    0.02      1177.0     609.0      568.0   
4  California                    0.26      1096.0     555.0      541.0   

   % Asian  % Black  % Hispanic  % White  % 

In [18]:
import pdfplumber
import pandas as pd
import os
import re

# Directory containing your downloaded state PDFs
pdf_dir = './state_pdfs'
all_state_data = []

def robust_clean(value, is_percentage=True):
    """Extracts numbers or percentages from messy PDF strings."""
    if not value or value == "0.0" or value == "0%":
        return 0.0
    val_str = str(value).replace(',', '')
    if is_percentage:
        match = re.search(r'(\d+)%', val_str)
        if match:
            return float(match.group(1)) / 100.0
    else:
        # For taker counts or scores, find the digits
        match = re.search(r'(\d+)', val_str)
        if match:
            return float(match.group(1))
    return 0.0

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    
    data = {
        'State': state_name,
        # Participation Expansion
        'SAT Takers': 0, 'High School Graduates': 0, 'SAT Participation Rate': 0.0,
        # Performance
        'Mean Total': 0, 'Mean ERW': 0, 'Mean Math': 0,
        # Demographics Expansion
        '% American Indian': 0.0, '% Asian': 0.0, '% Black': 0.0, '% Hispanic': 0.0, 
        '% Native Hawaiian': 0.0, '% White': 0.0, '% Two or More Races': 0.0, '% No Response Race': 0.0,
        # Parental Education Expansion
        '% No HS Diploma': 0.0, '% HS Diploma': 0.0, '% Associate Degree': 0.0, 
        '% Bachelor Degree': 0.0, '% Graduate Degree': 0.0, '% No Response Education': 0.0
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            p4_text = pdf.pages[3].extract_text() if len(pdf.pages) > 3 else ""
            p5_text = pdf.pages[4].extract_text() if len(pdf.pages) > 4 else ""
            
            # --- 1. PAGE 4: PARTICIPATION DATA ---
            # Extract SAT Takers and HS Graduates
            takers_match = re.search(r'(\d{1,3}(?:,\d{3})*)\s+SAT Takers', p4_text)
            if takers_match: data['SAT Takers'] = int(takers_match.group(1).replace(',', ''))
            
            grads_match = re.search(r'(\d{1,3}(?:,\d{3})*)\s+High School Graduates', p4_text)
            if grads_match: data['High School Graduates'] = int(grads_match.group(1).replace(',', ''))

            part_match = re.search(r'SAT Participation Rate\s+(\d+%)', p4_text)
            if part_match: data['SAT Participation Rate'] = robust_clean(part_match.group(1))

            # --- 2. PAGE 5: MEAN SCORES ---
            score_match = re.search(r'Total\s+[\d,]+\s+(\d{3,4})\s+(\d{2,3})\s+(\d{2,3})', p5_text)
            if score_match:
                data['Mean Total'], data['Mean ERW'], data['Mean Math'] = map(float, score_match.groups())

            # --- 3. PAGE 5: TABLES (DEMOGRAPHICS & EDUCATION) ---
            tables = pdf.pages[4].extract_tables()
            for table in tables:
                for row in table:
                    row_str = " ".join([str(c) for c in row if c]).lower()
                    
                    # Demographics
                    if 'american indian' in row_str: data['% American Indian'] = robust_clean(row_str)
                    if 'asian' in row_str: data['% Asian'] = robust_clean(row_str)
                    if 'black' in row_str or 'african american' in row_str: data['% Black'] = robust_clean(row_str)
                    if 'hispanic' in row_str or 'latino' in row_str: data['% Hispanic'] = robust_clean(row_str)
                    if 'native hawaiian' in row_str: data['% Native Hawaiian'] = robust_clean(row_str)
                    if 'white' in row_str: data['% White'] = robust_clean(row_str)
                    if 'two or more' in row_str: data['% Two or More Races'] = robust_clean(row_str)
                    
                    # Education
                    if 'no high school diploma' in row_str: data['% No HS Diploma'] = robust_clean(row_str)
                    if 'high school diploma' in row_str and 'no' not in row_str: data['% HS Diploma'] = robust_clean(row_str)
                    if 'associate' in row_str: data['% Associate Degree'] = robust_clean(row_str)
                    if 'bachelor' in row_str: data['% Bachelor Degree'] = robust_clean(row_str)
                    if 'graduate' in row_str: data['% Graduate Degree'] = robust_clean(row_str)

            # --- 4. SPECIAL HANDLING: "NO RESPONSE" ---
            # "No Response" appears in both tables, so we use proximity to the section headers
            race_section = p5_text.split("Race / Ethnicity")[1].split("Gender")[0] if "Race / Ethnicity" in p5_text else ""
            edu_section = p5_text.split("Highest Parental Education Level")[1] if "Highest Parental Education Level" in p5_text else ""
            
            nr_race = re.search(r'No Response\s+[\d,]+\s+(\d+%)', race_section)
            if nr_race: data['% No Response Race'] = robust_clean(nr_race.group(1))
            
            nr_edu = re.search(r'No Response\s+[\d,]+\s+(\d+%)', edu_section)
            if nr_edu: data['% No Response Education'] = robust_clean(nr_edu.group(1))

    except Exception as e:
        print(f"Error in {state_name}: {e}")
            
    return data

# --- Execution ---
print("Extracting expanded dataset...")
for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data)
df.to_csv('sat_expanded_dataset_2025.csv', index=False)
print("\nSuccess! Expanded dataset saved to CSV.")
print(df.head())

Extracting expanded dataset...

Success! Expanded dataset saved to CSV.
        State  SAT Takers  High School Graduates  SAT Participation Rate  \
0     Alabama           0                     25                    0.03   
1      Alaska           0                     25                    0.27   
2     Arizona           0                     25                    0.10   
3    Arkansas           0                     25                    0.02   
4  California           0                     25                    0.26   

   Mean Total  Mean ERW  Mean Math  % American Indian  % Asian  % Black  ...  \
0      1172.0     601.0      570.0                0.0     0.15      0.0  ...   
1      1097.0     567.0      530.0                0.0     0.06      0.0  ...   
2      1194.0     606.0      588.0                0.0     0.16      0.0  ...   
3      1177.0     609.0      568.0                0.0     0.15      0.0  ...   
4      1096.0     555.0      541.0                0.0     0.22      0.0

In [19]:
import pdfplumber
import pandas as pd
import os
import re

pdf_dir = './state_pdfs'
all_state_data = []

def robust_clean(value, is_percentage=True):
    """Extracts numbers or percentages from messy PDF strings."""
    if not value or value == "0.0" or value == "0%":
        return 0.0
    val_str = str(value).replace(',', '')
    if is_percentage:
        # Search for a number followed by a percent sign
        match = re.search(r'(\d+)%', val_str)
        if match:
            return float(match.group(1)) / 100.0
    else:
        # For counts or scores, find the first set of digits
        match = re.search(r'(\d+)', val_str)
        if match:
            return float(match.group(1))
    return 0.0

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    
    data = {
        'State': state_name,
        'SAT Takers': 0, 'High School Graduates': 0, 'SAT Participation Rate': 0.0,
        'Mean Total': 0, 'Mean ERW': 0, 'Mean Math': 0,
        '% American Indian': 0.0, '% Asian': 0.0, '% Black': 0.0, '% Hispanic': 0.0, 
        '% Native Hawaiian': 0.0, '% White': 0.0, '% Two or More Races': 0.0, '% No Response Race': 0.0,
        '% No HS Diploma': 0.0, '% HS Diploma': 0.0, '% Associate Degree': 0.0, 
        '% Bachelor Degree': 0.0, '% Graduate Degree': 0.0, '% No Response Education': 0.0
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            p4_text = pdf.pages[3].extract_text() if len(pdf.pages) > 3 else ""
            p5_text = pdf.pages[4].extract_text() if len(pdf.pages) > 4 else ""
            
            # --- 1. PAGE 4: PARTICIPATION (Regex Scan) ---
            # Extract Takers and Grads using numbers that appear BEFORE the label
            takers_match = re.search(r'([\d,]+)\s+SAT Takers', p4_text)
            if takers_match: data['SAT Takers'] = int(takers_match.group(1).replace(',', ''))
            
            grads_match = re.search(r'([\d,]+)\s+High School Graduates', p4_text)
            if grads_match: data['High School Graduates'] = int(grads_match.group(1).replace(',', ''))

            part_match = re.search(r'SAT Participation Rate\s+(\d+%)', p4_text)
            if part_match: data['SAT Participation Rate'] = robust_clean(part_match.group(1))

            # --- 2. PAGE 5: SCORES (Regex Scan) ---
            score_match = re.search(r'Total\s+[\d,]+\s+(\d{3,4})\s+(\d{2,3})\s+(\d{2,3})', p5_text)
            if score_match:
                data['Mean Total'], data['Mean ERW'], data['Mean Math'] = map(float, score_match.groups())

            # --- 3. PAGE 5: DEMOGRAPHICS & EDUCATION (Deep Keyword Scan) ---
            # We split the text into lines and check each line for keywords and percentages
            lines = p5_text.split('\n')
            
            for line in lines:
                l = line.lower()
                # Demographics
                if 'american indian' in l: data['% American Indian'] = robust_clean(line)
                if 'asian' in l: data['% Asian'] = robust_clean(line)
                if 'black' in l or 'african american' in l: data['% Black'] = robust_clean(line)
                if 'hispanic' in l or 'latino' in l: data['% Hispanic'] = robust_clean(line)
                if 'native hawaiian' in l: data['% Native Hawaiian'] = robust_clean(line)
                if 'white' in l: data['% White'] = robust_clean(line)
                if 'two or more' in l: data['% Two or More Races'] = robust_clean(line)
                
                # Education
                if 'no high school diploma' in l: data['% No HS Diploma'] = robust_clean(line)
                elif 'high school diploma' in l: data['% HS Diploma'] = robust_clean(line)
                if 'associate' in l: data['% Associate Degree'] = robust_clean(line)
                if 'bachelor' in l: data['% Bachelor Degree'] = robust_clean(line)
                if 'graduate' in l: data['% Graduate Degree'] = robust_clean(line)

            # --- 4. NO RESPONSE (Context-Aware Scan) ---
            # Race No Response usually comes before "Gender"
            if "Race / Ethnicity" in p5_text:
                race_part = p5_text.split("Race / Ethnicity")[1].split("Gender")[0]
                nr_match = re.search(r'No Response\s+[\d,]+\s+(\d+%)', race_part)
                if nr_match: data['% No Response Race'] = robust_clean(nr_match.group(1))
            
            # Education No Response usually comes after "Graduate Degree"
            if "Highest Parental Education Level" in p5_text:
                edu_part = p5_text.split("Highest Parental Education Level")[1]
                nr_match = re.search(r'No Response\s+[\d,]+\s+(\d+%)', edu_part)
                if nr_match: data['% No Response Education'] = robust_clean(nr_match.group(1))

    except Exception as e:
        print(f"Error in {state_name}: {e}")
            
    return data

# --- Execution ---
print("Extracting full dataset using Deep Text Scanning...")
for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data)
df.to_csv('sat_final_complete_2025.csv', index=False)
print("\nProcess Complete. File: 'sat_final_complete_2025.csv'")
print(df.head())

Extracting full dataset using Deep Text Scanning...

Process Complete. File: 'sat_final_complete_2025.csv'
        State  SAT Takers  High School Graduates  SAT Participation Rate  \
0     Alabama           0                   2025                    0.03   
1      Alaska           0                   2025                    0.27   
2     Arizona           0                   2025                    0.10   
3    Arkansas           0                   2025                    0.02   
4  California           0                   2025                    0.26   

   Mean Total  Mean ERW  Mean Math  % American Indian  % Asian  % Black  ...  \
0      1172.0     601.0      570.0               0.00     0.15     0.18  ...   
1      1097.0     567.0      530.0               0.05     0.06     0.02  ...   
2      1194.0     606.0      588.0               0.01     0.16     0.04  ...   
3      1177.0     609.0      568.0               0.01     0.15     0.08  ...   
4      1096.0     555.0      541.0  

In [20]:
import pdfplumber
import pandas as pd
import os
import re

pdf_dir = './state_pdfs'
all_state_data = []

def robust_clean(value, is_percentage=True):
    if not value or value == "0.0" or value == "0%":
        return 0.0
    val_str = str(value).replace(',', '')
    if is_percentage:
        # Match only the first % found in the string
        match = re.search(r'(\d+)%', val_str)
        if match:
            return float(match.group(1)) / 100.0
    else:
        # Match only digits (handling potential superscripts by taking the first group)
        match = re.search(r'(\d+)', val_str)
        if match:
            return float(match.group(1))
    return 0.0

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    
    data = {
        'State': state_name,
        'SAT Takers': 0, 'High School Graduates': 0, 'SAT Participation Rate': 0.0,
        'Mean Total': 0, 'Mean ERW': 0, 'Mean Math': 0,
        '% American Indian': 0.0, '% Asian': 0.0, '% Black': 0.0, '% Hispanic': 0.0, 
        '% Native Hawaiian': 0.0, '% White': 0.0, '% Two or More Races': 0.0, '% No Response Race': 0.0,
        '% No HS Diploma': 0.0, '% HS Diploma': 0.0, '% Associate Degree': 0.0, 
        '% Bachelor Degree': 0.0, '% Graduate Degree': 0.0, '% No Response Education': 0.0
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            p4_text = pdf.pages[3].extract_text() if len(pdf.pages) > 3 else ""
            p5_text = pdf.pages[4].extract_text() if len(pdf.pages) > 4 else ""
            
            # --- 1. PAGE 4: IMPROVED PARTICIPATION SCAN ---
            # We look for a number that is NOT 2025, followed by the labels (handling superscripts)
            # Pattern: Matches digits/commas, ignores if it's exactly 2025, looks for 'SAT Takers'
            takers_search = re.findall(r'([\d,]+)\s+SAT Takers', p4_text)
            for val in takers_search:
                clean_val = val.replace(',', '')
                if clean_val != "2025":
                    data['SAT Takers'] = int(clean_val)
                    break
            
            grads_search = re.findall(r'([\d,]+)\s+High School Graduates', p4_text)
            for val in grads_search:
                clean_val = val.replace(',', '')
                if clean_val != "2025":
                    data['High School Graduates'] = int(clean_val)
                    break

            part_match = re.search(r'SAT Participation Rate\s+(\d+%)', p4_text)
            if part_match: data['SAT Participation Rate'] = robust_clean(part_match.group(1))

            # --- 2. PAGE 5: PERFORMANCE ---
            score_match = re.search(r'Total\s+[\d,]+\s+(\d{3,4})\s+(\d{2,3})\s+(\d{2,3})', p5_text)
            if score_match:
                data['Mean Total'], data['Mean ERW'], data['Mean Math'] = map(float, score_match.groups())

            # --- 3. PAGE 5: DEMOGRAPHICS & EDUCATION (Line Scan) ---
            lines = p5_text.split('\n')
            for line in lines:
                l = line.lower()
                if 'american indian' in l: data['% American Indian'] = robust_clean(line)
                if 'asian' in l: data['% Asian'] = robust_clean(line)
                if 'black' in l or 'african american' in l: data['% Black'] = robust_clean(line)
                if 'hispanic' in l or 'latino' in l: data['% Hispanic'] = robust_clean(line)
                if 'native hawaiian' in l: data['% Native Hawaiian'] = robust_clean(line)
                if 'white' in l: data['% White'] = robust_clean(line)
                if 'two or more' in l: data['% Two or More Races'] = robust_clean(line)
                
                if 'no high school diploma' in l: data['% No HS Diploma'] = robust_clean(line)
                elif 'high school diploma' in l: data['% HS Diploma'] = robust_clean(line)
                if 'associate' in l: data['% Associate Degree'] = robust_clean(line)
                if 'bachelor' in l: data['% Bachelor Degree'] = robust_clean(line)
                if 'graduate' in l: data['% Graduate Degree'] = robust_clean(line)

            # --- 4. NO RESPONSE (Proximity Search) ---
            # Race No Response: First "No Response" after the Race header
            if "Race / Ethnicity" in p5_text:
                race_sec = p5_text.split("Race / Ethnicity")[1].split("Gender")[0]
                nr_race = re.search(r'No Response\s+[\d,]+\s+(\d+%)', race_sec)
                if nr_race: data['% No Response Race'] = robust_clean(nr_race.group(1))
            
            # Education No Response: Find the percentage that appears after "Graduate Degree"
            if "Highest Parental Education Level" in p5_text:
                edu_sec = p5_text.split("Highest Parental Education Level")[1]
                # In CB reports, the No Response row is the very last one in this section
                nr_edu_matches = re.findall(r'No Response\s+[\d,]+\s+(\d+%)', edu_sec)
                if nr_edu_matches:
                    data['% No Response Education'] = robust_clean(nr_edu_matches[-1])

    except Exception as e:
        print(f"Error in {state_name}: {e}")
            
    return data

# --- Main Execution ---
print("Running Final Extraction pass...")
for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data)
df.to_csv('sat_final_master_2025.csv', index=False)
print("\nSuccess! Data exported to 'sat_final_master_2025.csv'")
print(df[['State', 'SAT Takers', 'High School Graduates', '% No Response Education']].head())

Running Final Extraction pass...

Success! Data exported to 'sat_final_master_2025.csv'
        State  SAT Takers  High School Graduates  % No Response Education
0     Alabama           0                      0                      0.0
1      Alaska           0                      0                      0.0
2     Arizona           0                      0                      0.0
3    Arkansas           0                      0                      0.0
4  California           0                      0                      0.0


In [21]:
import pdfplumber
import pandas as pd
import os
import re

pdf_dir = './state_pdfs'
all_state_data = []

def robust_clean(value, is_percentage=True):
    """Extracts numbers or percentages from messy PDF strings with varied spacing."""
    if not value or value == "0.0" or value == "0%":
        return 0.0
    # Normalize spaces and remove commas
    val_str = re.sub(r'\s+', ' ', str(value)).replace(',', '')
    if is_percentage:
        # Search for a number followed by a percent sign anywhere in the string
        match = re.search(r'(\d+)%', val_str)
        if match:
            return float(match.group(1)) / 100.0
    else:
        # Match only digits
        match = re.search(r'(\d+)', val_str)
        if match:
            return float(match.group(1))
    return 0.0

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    
    data = {
        'State': state_name,
        'SAT Takers': 0, 'High School Graduates': 0, 'SAT Participation Rate': 0.0,
        'Mean Total': 0, 'Mean ERW': 0, 'Mean Math': 0,
        '% American Indian': 0.0, '% Asian': 0.0, '% Black': 0.0, '% Hispanic': 0.0, 
        '% Native Hawaiian': 0.0, '% White': 0.0, '% Two or More Races': 0.0, '% No Response Race': 0.0,
        '% No HS Diploma': 0.0, '% HS Diploma': 0.0, '% Associate Degree': 0.0, 
        '% Bachelor Degree': 0.0, '% Graduate Degree': 0.0, '% No Response Education': 0.0
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            p4_text = pdf.pages[3].extract_text() if len(pdf.pages) > 3 else ""
            p5_text = pdf.pages[4].extract_text() if len(pdf.pages) > 4 else ""
            
            # --- 1. PAGE 4: PARTICIPATION (Flexible Space Regex) ---
            # Handles "SAT  Takers¹ 11,249" or "11,249 SAT Takers"
            takers_m = re.search(r'SAT\s+Takers\d*[\s\D]+([\d,]+)', p4_text, re.IGNORECASE)
            if takers_m: data['SAT Takers'] = int(takers_m.group(1).replace(',', ''))
            
            grads_m = re.search(r'High\s+School\s+Graduates\d*[\s\D]+([\d,]+)', p4_text, re.IGNORECASE)
            if grads_m: data['High School Graduates'] = int(grads_m.group(1).replace(',', ''))

            part_m = re.search(r'SAT\s+Participation\s+Rate\s+(\d+%)', p4_text, re.IGNORECASE)
            if part_m: data['SAT Participation Rate'] = robust_clean(part_m.group(1))

            # --- 2. PAGE 5: SCORES ---
            score_m = re.search(r'Total\s+[\d,]+\s+(\d{3,4})\s+(\d{2,3})\s+(\d{2,3})', p5_text)
            if score_m:
                data['Mean Total'], data['Mean ERW'], data['Mean Math'] = map(float, score_m.groups())

            # --- 3. PAGE 5: DEMOGRAPHICS & EDUCATION (Line-by-Line) ---
            lines = p5_text.split('\n')
            for line in lines:
                l = re.sub(r'\s+', ' ', line.lower()) # Normalize line spacing
                
                # Demographics
                if 'american indian' in l: data['% American Indian'] = robust_clean(line)
                if 'asian' in l: data['% Asian'] = robust_clean(line)
                if 'black' in l or 'african american' in l: data['% Black'] = robust_clean(line)
                if 'hispanic' in l or 'latino' in l: data['% Hispanic'] = robust_clean(line)
                if 'native hawaiian' in l: data['% Native Hawaiian'] = robust_clean(line)
                if 'white' in l: data['% White'] = robust_clean(line)
                if 'two or more' in l: data['% Two or More Races'] = robust_clean(line)
                
                # Education
                if 'no high school diploma' in l: data['% No HS Diploma'] = robust_clean(line)
                elif 'high school diploma' in l: data['% HS Diploma'] = robust_clean(line)
                if 'associate' in l: data['% Associate Degree'] = robust_clean(line)
                if 'bachelor' in l: data['% Bachelor Degree'] = robust_clean(line)
                if 'graduate' in l: data['% Graduate Degree'] = robust_clean(line)

            # --- 4. NO RESPONSE (Context-Specific Search) ---
            # Extract "No Response" based on section headers to avoid mixing Race vs Education
            if "Race / Ethnicity" in p5_text:
                race_section = p5_text.split("Race / Ethnicity")[1].split("Gender")[0]
                nr_m = re.search(r'No\s+Response\s+[\d,]+\s+(\d+%)', race_section, re.IGNORECASE)
                if nr_m: data['% No Response Race'] = robust_clean(nr_m.group(1))
            
            if "Level of Parental Education" in p5_text:
                edu_section = p5_text.split("Level of Parental Education")[1]
                # Education 'No Response' is the last row in this section
                nr_matches = re.findall(r'No\s+Response\s+[\d,]+\s+(\d+%)', edu_section, re.IGNORECASE)
                if nr_matches: data['% No Response Education'] = robust_clean(nr_matches[-1])

    except Exception as e:
        print(f"Skipping {state_name}: {e}")
            
    return data

# --- Main Logic ---
print("Extracting full master dataset...")
for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data)
df.to_csv('sat_final_master_2025.csv', index=False)
print("\nSuccess! Dataset created with all fields.")
print(df[['State', 'SAT Takers', 'High School Graduates', '% No Response Education']].head())

Extracting full master dataset...

Success! Dataset created with all fields.
        State  SAT Takers  High School Graduates  % No Response Education
0     Alabama        1556                  55021                     0.08
1      Alaska        2332                   8722                     0.14
2     Arizona        8327                  85639                     0.11
3    Arkansas         783                  36169                     0.08
4  California      123259                 469214                     0.19


In [23]:
import pdfplumber
import pandas as pd
import os
import re

# Directory containing your downloaded state PDFs
pdf_dir = './state_pdfs'
all_state_data = []

def extract_val(line, is_percentage=True):
    """
    Finds the first percentage or number in a string.
    Normalizes spacing to handle 'ghost' characters.
    """
    if not line: return 0.0
    line = re.sub(r'\s+', ' ', line).strip()
    
    if is_percentage:
        match = re.search(r'(\d+)%', line)
        return float(match.group(1)) / 100.0 if match else 0.0
    else:
        # For counts: find numbers that are not '2025'
        nums = re.findall(r'[\d,]+', line)
        for n in nums:
            val = n.replace(',', '')
            if val != "2025" and len(val) > 1:
                return float(val)
    return 0.0

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    
    data = {
        'State': state_name,
        # Participation
        'SAT Takers': 0, 'High School Graduates': 0, 'SAT Participation Rate': 0.0,
        # Performance
        'Mean Total': 0, 'Mean ERW': 0, 'Mean Math': 0,
        # Demographics
        '% American Indian': 0.0, '% Asian': 0.0, '% Black': 0.0, '% Hispanic': 0.0, 
        '% Native Hawaiian': 0.0, '% White': 0.0, '% Two or More Races': 0.0, '% No Response Race': 0.0,
        # Education
        '% No HS Diploma': 0.0, '% HS Diploma': 0.0, '% Associate Degree': 0.0, 
        '% Bachelor Degree': 0.0, '% Graduate Degree': 0.0, '% No Response Education': 0.0
    }
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            p4_text = pdf.pages[3].extract_text() if len(pdf.pages) > 3 else ""
            p5_text = pdf.pages[4].extract_text() if len(pdf.pages) > 4 else ""
            
            # --- Page 4: Takers and Graduates ---
            # We look for the label, skip any superscripts/spaces, then grab the digits
            takers_m = re.search(r'SAT\s+Takers\D+([\d,]+)', p4_text, re.IGNORECASE)
            if takers_m: data['SAT Takers'] = int(takers_m.group(1).replace(',', ''))
            
            grads_m = re.search(r'High\s+School\s+Graduates\D+([\d,]+)', p4_text, re.IGNORECASE)
            if grads_m: data['High School Graduates'] = int(grads_m.group(1).replace(',', ''))
            
            part_m = re.search(r'Participation\s+Rate\s+(\d+%)', p4_text, re.IGNORECASE)
            if part_m: data['SAT Participation Rate'] = extract_val(part_m.group(1))

            # --- Page 5: Performance ---
            score_m = re.search(r'Total\s+[\d,]+\s+(\d{3,4})\s+(\d{2,3})\s+(\d{2,3})', p5_text)
            if score_m:
                data['Mean Total'], data['Mean ERW'], data['Mean Math'] = map(float, score_m.groups())

            # --- Page 5: Demographic & Education Sections ---
            # Split the text by headers to ensure "No Response" is mapped correctly
            sections = re.split(r'(Race / Ethnicity|Gender|First Language|Highest Level of Parental Education)', p5_text)
            
            # Demographics Section
            if 'Race / Ethnicity' in sections:
                race_text = sections[sections.index('Race / Ethnicity') + 1]
                for line in race_text.split('\n'):
                    l = line.lower()
                    if 'american indian' in l: data['% American Indian'] = extract_val(line)
                    elif 'asian' in l: data['% Asian'] = extract_val(line)
                    elif 'black' in l or 'african american' in l: data['% Black'] = extract_val(line)
                    elif 'hispanic' in l or 'latino' in l: data['% Hispanic'] = extract_val(line)
                    elif 'native hawaiian' in l: data['% Native Hawaiian'] = extract_val(line)
                    elif 'white' in l: data['% White'] = extract_val(line)
                    elif 'two or more' in l: data['% Two or More Races'] = extract_val(line)
                    elif 'no response' in l: data['% No Response Race'] = extract_val(line)

            # Education Section
            if 'Highest Level of Parental Education' in sections:
                edu_text = sections[sections.index('Highest Level of Parental Education') + 1]
                for line in edu_text.split('\n'):
                    l = line.lower()
                    if 'no high school diploma' in l: data['% No HS Diploma'] = extract_val(line)
                    elif 'high school diploma' in l: data['% HS Diploma'] = extract_val(line)
                    elif 'associate' in l: data['% Associate Degree'] = extract_val(line)
                    elif 'bachelor' in l: data['% Bachelor Degree'] = extract_val(line)
                    elif 'graduate' in l: data['% Graduate Degree'] = extract_val(line)
                    elif 'no response' in l: data['% No Response Education'] = extract_val(line)

    except Exception as e:
        print(f"Skipping {state_name}: {e}")
            
    return data

# --- Run ---
print("Extracting full master dataset...")
for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data)
df.to_csv('sat_final_master_2025.csv', index=False)
print("\nSuccess! Dataset saved to 'sat_final_master_2025.csv'.")
print(df[['State', 'SAT Takers', 'High School Graduates', '% No Response Education']].head())

Extracting full master dataset...

Success! Dataset saved to 'sat_final_master_2025.csv'.
        State  SAT Takers  High School Graduates  % No Response Education
0     Alabama        1556                  55021                     0.08
1      Alaska        2332                   8722                     0.14
2     Arizona        8327                  85639                     0.11
3    Arkansas         783                  36169                     0.08
4  California      123259                 469214                     0.19


In [24]:
import pdfplumber
import pandas as pd
import os
import re

pdf_dir = './state_pdfs'
all_state_data = []

def extract_val(line, is_percentage=True):
    """Handles percentages, counts, and scores with high precision."""
    if not line: return 0.0
    # Clean ghost spaces and symbols
    line = re.sub(r'\s+', ' ', line).replace(',', '').replace('$', '')
    
    if is_percentage:
        match = re.search(r'(\d+)%', line)
        return float(match.group(1)) / 100.0 if match else 0.0
    else:
        # Find numeric values (ignoring the year 2025)
        nums = re.findall(r'\d+', line)
        for n in nums:
            if n != "2025" and len(n) > 1:
                return float(n)
    return 0.0

def get_row_stats(text_block, keyword):
    """Extracts a full row of stats: [Count, Percent, Total, ERW, Math]"""
    lines = text_block.split('\n')
    for line in lines:
        if keyword.lower() in line.lower():
            # Extract all numbers/percentages from the line
            parts = re.findall(r'([\d,]+%?)', line)
            # Standard CB layout: Count, Percent, Total, ERW, Math
            if len(parts) >= 5:
                count = float(parts[0].replace(',', ''))
                pct = float(parts[1].replace('%', '')) / 100.0
                total = float(parts[2].replace(',', ''))
                erw = float(parts[3].replace(',', ''))
                math = float(parts[4].replace(',', ''))
                return count, pct, total, erw, math
    return 0, 0.0, 0, 0, 0

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    
    data = {'State': state_name}
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # Combine text for easier searching
            p4_text = pdf.pages[3].extract_text() if len(pdf.pages) > 3 else ""
            p5_text = pdf.pages[4].extract_text() if len(pdf.pages) > 4 else ""
            # School location is often on Page 6 or 7
            full_text = "\n".join([p.extract_text() for p in pdf.pages[3:8]])
            
            # 1. PARTICIPATION (Page 4)
            takers_m = re.search(r'SAT\s+Takers\D+([\d,]+)', p4_text, re.IGNORECASE)
            data['SAT Takers'] = int(takers_m.group(1).replace(',', '')) if takers_m else 0
            
            grads_m = re.search(r'High\s+School\s+Graduates\D+([\d,]+)', p4_text, re.IGNORECASE)
            data['HS Grads'] = int(grads_m.group(1).replace(',', '')) if grads_m else 0

            # 2. PERFORMANCE & GENDER (Page 5)
            sections = re.split(r'(Race / Ethnicity|Gender|Parental Education|Family Income|School Location)', full_text)
            
            # Gender Extraction
            if 'Gender' in sections:
                gender_sec = sections[sections.index('Gender') + 1]
                data['Female_Takers'], data['Female_Pct'], data['Female_Total'], data['Female_ERW'], data['Female_Math'] = get_row_stats(gender_sec, 'Female')
                data['Male_Takers'], data['Male_Pct'], data['Male_Total'], data['Male_ERW'], data['Male_Math'] = get_row_stats(gender_sec, 'Male')

            # 3. INCOME QUINTILES
            if 'Family Income' in sections:
                inc_sec = sections[sections.index('Family Income') + 1]
                data['% Inc_Lowest'] = extract_val(re.search(r'0\s*-\s*59,669', inc_sec).group(0) if re.search(r'0\s*-\s*59,669', inc_sec) else "")
                data['% Inc_2nd_Low'] = extract_val(re.search(r'59,670\s*-\s*76,797', inc_sec).group(0) if re.search(r'59,670\s*-\s*76,797', inc_sec) else "")
                data['% Inc_Middle'] = extract_val(re.search(r'76,798\s*-\s*95,223', inc_sec).group(0) if re.search(r'76,798\s*-\s*95,223', inc_sec) else "")
                data['% Inc_2nd_High'] = extract_val(re.search(r'95,224\s*-\s*124,913', inc_sec).group(0) if re.search(r'95,224\s*-\s*124,913', inc_sec) else "")
                data['% Inc_Highest'] = extract_val(re.search(r'124,914', inc_sec).group(0) if re.search(r'124,914', inc_sec) else "")

            # 4. SCHOOL LOCATION
            if 'School Location' in sections:
                loc_sec = sections[sections.index('School Location') + 1]
                data['City_N'], data['City_Pct'], data['City_Total'], data['City_ERW'], data['City_Math'] = get_row_stats(loc_sec, 'City')
                data['Suburb_N'], data['Suburb_Pct'], data['Suburb_Total'], data['Suburb_ERW'], data['Suburb_Math'] = get_row_stats(loc_sec, 'Suburb')
                data['Town_N'], data['Town_Pct'], data['Town_Total'], data['Town_ERW'], data['Town_Math'] = get_row_stats(loc_sec, 'Town/Rural')
                data['Unknown_Loc_N'], data['Unknown_Loc_Pct'], data['Unknown_Loc_Total'], data['Unknown_Loc_ERW'], data['Unknown_Loc_Math'] = get_row_stats(loc_sec, 'Unknown')

    except Exception as e:
        print(f"Error in {state_name}: {e}")
            
    return data

# --- Execution ---
print("Extracting full modeling dataset...")
for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data)
df.to_csv('sat_modeling_master_2025.csv', index=False)
print("Success! Dataset generated with Income, Gender, and Location fields.")

Extracting full modeling dataset...
Success! Dataset generated with Income, Gender, and Location fields.


In [25]:
import pdfplumber
import pandas as pd
import os
import re

pdf_dir = './state_pdfs'
all_state_data = []

def extract_pct_from_line(text, pattern):
    """
    Finds a line matching a pattern and extracts the first percentage found.
    """
    if not text: return 0.0
    # Normalize spaces to handle PDF ghost characters
    normalized_text = re.sub(r'\s+', ' ', text)
    match = re.search(pattern, normalized_text, re.IGNORECASE)
    if match:
        # Get the text from the start of the match to the end of that line
        line_fragment = normalized_text[match.start():match.start()+100]
        pct_match = re.search(r'(\d+)%', line_fragment)
        if pct_match:
            return float(pct_match.group(1)) / 100.0
    return 0.0

def get_row_stats(text_block, keyword):
    """Extracts [Count, Percent, Total, ERW, Math] for a labeled row."""
    if not text_block: return 0, 0.0, 0, 0, 0
    lines = text_block.split('\n')
    for line in lines:
        if keyword.lower() in line.lower():
            # Extract all numeric/percentage chunks
            parts = re.findall(r'([\d,]+%?)', line)
            if len(parts) >= 5:
                try:
                    count = float(parts[0].replace(',', ''))
                    pct = float(parts[1].replace('%', '')) / 100.0
                    total = float(parts[2].replace(',', ''))
                    erw = float(parts[3].replace(',', ''))
                    math = float(parts[4].replace(',', ''))
                    return count, pct, total, erw, math
                except: continue
    return 0, 0.0, 0, 0, 0

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    data = {'State': state_name}
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # Gather text from relevant pages
            p4_text = pdf.pages[3].extract_text() if len(pdf.pages) > 3 else ""
            # Scanning multiple pages as 'School Location' and 'Income' can shift
            full_text = "\n".join([p.extract_text() for p in pdf.pages[3:9] if p.extract_text()])
            
            # --- 1. PARTICIPATION & TOTAL MEANS ---
            takers_m = re.search(r'SAT\s+Takers\D+([\d,]+)', p4_text, re.IGNORECASE)
            data['SAT Takers'] = int(takers_m.group(1).replace(',', '')) if takers_m else 0
            
            grads_m = re.search(r'High\s+School\s+Graduates\D+([\d,]+)', p4_text, re.IGNORECASE)
            data['HS Grads'] = int(grads_m.group(1).replace(',', '')) if grads_m else 0

            # --- 2. SECTIONAL BREAKDOWN ---
            # We split by major headers to prevent cross-contamination of "No Response" or "Unknown"
            sections = re.split(r'(Race / Ethnicity|Gender|Parental Education|Family Income|School Location)', full_text)
            
            # GENDER
            if 'Gender' in sections:
                gender_sec = sections[sections.index('Gender') + 1]
                data['F_N'], data['F_Pct'], data['F_Total'], data['F_ERW'], data['F_Math'] = get_row_stats(gender_sec, 'Female')
                data['M_N'], data['M_Pct'], data['M_Total'], data['M_ERW'], data['M_Math'] = get_row_stats(gender_sec, 'Male')

            # INCOME QUINTILES (The Fix)
            if 'Family Income' in sections:
                inc_sec = sections[sections.index('Family Income') + 1]
                # We search for the start of the range and grab the first % on that line
                data['Inc_Lowest_Pct'] = extract_pct_from_line(inc_sec, r'0\s*-\s*59,669')
                data['Inc_2nd_Low_Pct'] = extract_pct_from_line(inc_sec, r'59,670\s*-\s*76,797')
                data['Inc_Middle_Pct'] = extract_pct_from_line(inc_sec, r'76,798\s*-\s*95,223')
                data['Inc_2nd_High_Pct'] = extract_pct_from_line(inc_sec, r'95,224\s*-\s*124,913')
                data['Inc_Highest_Pct'] = extract_pct_from_line(inc_sec, r'124,914')
                data['Inc_Unknown_Pct'] = extract_pct_from_line(inc_sec, r'No Response|Unknown')

            # SCHOOL LOCATION
            if 'School Location' in sections:
                loc_sec = sections[sections.index('School Location') + 1]
                data['City_N'], data['City_Pct'], data['City_Tot'], data['City_ERW'], data['City_Math'] = get_row_stats(loc_sec, 'City')
                data['Sub_N'], data['Sub_Pct'], data['Sub_Tot'], data['Sub_ERW'], data['Sub_Math'] = get_row_stats(loc_sec, 'Suburb')
                data['Town_N'], data['Town_Pct'], data['Town_Tot'], data['Town_ERW'], data['Town_Math'] = get_row_stats(loc_sec, 'Town/Rural')
                data['Loc_Unk_N'], data['Loc_Unk_Pct'], data['Loc_Unk_Tot'], data['Loc_Unk_ERW'], data['Loc_Unk_Math'] = get_row_stats(loc_sec, 'Unknown')

    except Exception as e:
        print(f"Error in {state_name}: {e}")
            
    return data

# --- Run ---
print("Extracting full master dataset...")
for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data)
df.to_csv('sat_final_master_2025.csv', index=False)
print("\nProcess Complete. Check 'sat_final_master_2025.csv'")

Extracting full master dataset...

Process Complete. Check 'sat_final_master_2025.csv'


In [26]:
import pdfplumber
import pandas as pd
import os
import re

pdf_dir = './state_pdfs'
all_state_data = []

def get_row_stats_by_range(text_block, range_start_digit):
    """
    Finds a row by the unique starting number of the income range
    and extracts: Count, Percent, Total, ERW, Math.
    """
    if not text_block: return 0, 0.0, 0, 0, 0
    lines = text_block.split('\n')
    for line in lines:
        # Normalize line and look for the specific starting number of the range
        clean_line = re.sub(r'\s+', ' ', line).replace('$', '')
        if range_start_digit in clean_line:
            # Extract all numeric/percentage values
            parts = re.findall(r'([\d,]+%?)', line)
            # The label itself contains numbers, so we skip the first few 
            # to get to the actual data columns (Count, Pct, Total, ERW, Math)
            # College Board rows usually have 5 data points after the label
            data_parts = [p for p in parts if '%' in p or len(p.replace(',','')) >= 2]
            
            # We look for the first % sign as our anchor for the data columns
            for i, p in enumerate(data_parts):
                if '%' in p and i > 0:
                    try:
                        count = float(data_parts[i-1].replace(',', ''))
                        pct = float(p.replace('%', '')) / 100.0
                        total = float(data_parts[i+1].replace(',', ''))
                        erw = float(data_parts[i+2].replace(',', ''))
                        math = float(data_parts[i+3].replace(',', ''))
                        return count, pct, total, erw, math
                    except: continue
    return 0, 0.0, 0, 0, 0

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    data = {'State': state_name}
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            p4_text = pdf.pages[3].extract_text() if len(pdf.pages) > 3 else ""
            full_text = "\n".join([p.extract_text() for p in pdf.pages[3:9] if p.extract_text()])
            
            # 1. BASIC PARTICIPATION
            takers_m = re.search(r'SAT\s+Takers\D+([\d,]+)', p4_text, re.IGNORECASE)
            data['SAT Takers'] = int(takers_m.group(1).replace(',', '')) if takers_m else 0
            
            # 2. SECTIONAL SPLIT
            sections = re.split(r'(Race / Ethnicity|Gender|Parental Education|Family Income|School Location)', full_text)
            
            if 'Family Income' in sections:
                inc_sec = sections[sections.index('Family Income') + 1]
                
                # Mapping each Quintile to its unique starting number in the label
                income_map = {
                    'Inc_Lowest': '0',
                    'Inc_2ndLow': '59,670',
                    'Inc_Middle': '76,798',
                    'Inc_2ndHigh': '95,224',
                    'Inc_Highest': '124,914',
                    'Inc_Unknown': 'No Response'
                }
                
                for key, anchor in income_map.items():
                    if key == 'Inc_Unknown':
                        # Handle 'Unknown' using the standard row stats function
                        res = get_row_stats_by_range(inc_sec, anchor)
                    else:
                        res = get_row_stats_by_range(inc_sec, anchor)
                        
                    data[f'{key}_N'], data[f'{key}_Pct'], data[f'{key}_Tot'], data[f'{key}_ERW'], data[f'{key}_Math'] = res

            # 3. SCHOOL LOCATION (standard extraction)
            if 'School Location' in sections:
                loc_sec = sections[sections.index('School Location') + 1]
                for loc in ['City', 'Suburb', 'Town/Rural', 'Unknown']:
                    res = get_row_stats_by_range(loc_sec, loc)
                    data[f'{loc}_N'], data[f'{loc}_Pct'], data[f'{loc}_Tot'], data[f'{loc}_ERW'], data[f'{loc}_Math'] = res

    except Exception as e:
        print(f"Error in {state_name}: {e}")
            
    return data

# --- Run ---
print("Extracting detailed master dataset...")
for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data)
df.to_csv('sat_detailed_modeling_2025.csv', index=False)
print("\nSuccess! Full stats for Income Quintiles extracted.")

Extracting detailed master dataset...

Success! Full stats for Income Quintiles extracted.


In [27]:
import pdfplumber
import pandas as pd
import os
import re

pdf_dir = './state_pdfs'
all_state_data = []

def get_row_stats_by_range(text_block, anchor_keyword):
    """
    Finds a row by a keyword and extracts: Count, Percent, Total, ERW, Math.
    Uses the '%' sign as an anchor to distinguish labels from data.
    """
    if not text_block: return 0, 0.0, 0, 0, 0
    lines = text_block.split('\n')
    for line in lines:
        clean_line = re.sub(r'\s+', ' ', line).strip()
        if anchor_keyword.lower() in clean_line.lower():
            # Extract all numeric/percentage chunks
            parts = re.findall(r'([\d,]+%?)', clean_line)
            
            # Anchor on the percentage sign
            for i, p in enumerate(parts):
                if '%' in p and i > 0:
                    try:
                        count = float(parts[i-1].replace(',', ''))
                        pct = float(p.replace('%', '')) / 100.0
                        total = float(parts[i+1].replace(',', ''))
                        erw = float(parts[i+2].replace(',', ''))
                        math = float(parts[i+3].replace(',', ''))
                        return count, pct, total, erw, math
                    except (IndexError, ValueError):
                        continue
    return 0, 0.0, 0, 0, 0

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    data = {'State': state_name}
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            p4_text = pdf.pages[3].extract_text() if len(pdf.pages) > 3 else ""
            # Search across pages 5-8 where these tables usually live
            full_text = "\n".join([p.extract_text() for p in pdf.pages[4:8] if p.extract_text()])
            
            # --- 1. PARTICIPATION ---
            takers_m = re.search(r'SAT\s+Takers\D+([\d,]+)', p4_text, re.IGNORECASE)
            data['SAT Takers'] = int(takers_m.group(1).replace(',', '')) if takers_m else 0

            # --- 2. SECTIONAL ISOLATION ---
            # We split by major headers to prevent "No Response" in Income 
            # from being confused with "No Response" in Education or Language.
            sections = re.split(r'(Race / Ethnicity|Gender|Parental Education|Family Income|School Location)', full_text)
            
            # INCOME QUINTILES
            if 'Family Income' in sections:
                idx = sections.index('Family Income')
                # The text block immediately following the header
                inc_sec = sections[idx + 1] 
                
                income_map = {
                    'Inc_Lowest': '0 - 59,669',
                    'Inc_2ndLow': '59,670',
                    'Inc_Middle': '76,798',
                    'Inc_2ndHigh': '95,224',
                    'Inc_Highest': '124,914',
                    'Inc_Unknown': 'No Response' # This is the specific label for Unknown Income
                }
                
                for key, anchor in income_map.items():
                    res = get_row_stats_by_range(inc_sec, anchor)
                    data[f'{key}_N'], data[f'{key}_Pct'], data[f'{key}_Tot'], data[f'{key}_ERW'], data[f'{key}_Math'] = res

            # SCHOOL LOCATION
            if 'School Location' in sections:
                loc_idx = sections.index('School Location')
                loc_sec = sections[loc_idx + 1]
                for loc in ['City', 'Suburb', 'Town/Rural', 'Unknown']:
                    res = get_row_stats_by_range(loc_sec, loc)
                    data[f'{loc}_N'], data[f'{loc}_Pct'], data[f'{loc}_Tot'], data[f'{loc}_ERW'], data[f'{loc}_Math'] = res

    except Exception as e:
        print(f"Error in {state_name}: {e}")
            
    return data

# --- Run ---
print("Running final extraction for all variables...")
for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data)
df.to_csv('sat_final_complete_modeling_data.csv', index=False)
print("\nExtraction finished. 'Inc_Unknown' and 'Unknown' Location are now separated and populated.")

Running final extraction for all variables...

Extraction finished. 'Inc_Unknown' and 'Unknown' Location are now separated and populated.


In [28]:
import pdfplumber
import pandas as pd
import os
import re

pdf_dir = './state_pdfs'
all_state_data = []

def extract_row_data(text_block, search_pattern):
    """
    Finds a line by pattern and extracts: Count, Percent, Total, ERW, Math.
    This version is more aggressive about filtering out labels from data.
    """
    if not text_block: return 0, 0.0, 0, 0, 0
    lines = text_block.split('\n')
    
    for line in lines:
        # Normalize and remove $ for cleaner matching
        clean_line = re.sub(r'\s+', ' ', line).replace('$', '')
        
        if re.search(search_pattern, clean_line, re.IGNORECASE):
            # Find all chunks that are either a percentage or a multi-digit number
            # We filter out '2025' and single digits often used for superscripts
            parts = re.findall(r'([\d,]+%?)', line)
            data_candidates = []
            for p in parts:
                val = p.replace(',', '').replace('%', '')
                if val == "2025": continue
                # We need the count, pct, and scores. Scores are 3 digits, counts vary.
                data_candidates.append(p)
            
            # Anchor on the percentage sign to find the data columns
            for i, p in enumerate(data_candidates):
                if '%' in p and i > 0:
                    try:
                        # Row Structure: [Count] [Percent] [Total Mean] [ERW Mean] [Math Mean]
                        count = float(data_candidates[i-1].replace(',', ''))
                        pct = float(p.replace('%', '')) / 100.0
                        total = float(data_candidates[i+1].replace(',', ''))
                        erw = float(data_candidates[i+2].replace(',', ''))
                        math = float(data_candidates[i+3].replace(',', ''))
                        return count, pct, total, erw, math
                    except (IndexError, ValueError):
                        continue
    return 0, 0.0, 0, 0, 0

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    data = {'State': state_name}
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # Participation usually on Page 4
            p4_text = pdf.pages[3].extract_text() if len(pdf.pages) > 3 else ""
            # Income and Location usually on Page 5 or 6
            p5_8_text = "\n".join([p.extract_text() for p in pdf.pages[4:8] if p.extract_text()])
            
            # --- 1. PARTICIPATION ---
            takers_m = re.search(r'SAT\s+Takers\D+([\d,]+)', p4_text, re.IGNORECASE)
            data['SAT_Takers'] = int(takers_m.group(1).replace(',', '')) if takers_m else 0

            # --- 2. SECTIONAL ISOLATION ---
            # Isolate the Family Income section to avoid 'No Response' cross-talk
            inc_match = re.search(r'Family Income(.*?)(Gender|School Location|First Language)', p5_8_text, re.DOTALL | re.IGNORECASE)
            inc_sec = inc_match.group(1) if inc_match else ""
            
            # Isolate School Location
            loc_match = re.search(r'School Location(.*?)(AP Score|Instructional|$) ', p5_8_text, re.DOTALL | re.IGNORECASE)
            loc_sec = loc_match.group(1) if loc_match else ""

            # --- 3. INCOME EXTRACTION ---
            # Using specific numeric ranges to avoid superscript interference
            income_patterns = {
                'Inc_Lowest': r'0\s?-\s?59,669',
                'Inc_2ndLow': r'59,670\s?-\s?76,797',
                'Inc_Middle': r'76,798\s?-\s?95,223',
                'Inc_2ndHigh': r'95,224\s?-\s?124,913',
                'Inc_Highest': r'124,914',
                'Inc_Unknown': r'No Response'
            }
            
            for key, pattern in income_patterns.items():
                res = extract_row_data(inc_sec, pattern)
                data[f'{key}_N'], data[f'{key}_Pct'], data[f'{key}_Tot'], data[f'{key}_ERW'], data[f'{key}_Math'] = res

            # --- 4. LOCATION EXTRACTION ---
            location_patterns = {
                'City': r'City',
                'Suburb': r'Suburb',
                'Town_Rural': r'Town/Rural',
                'Loc_Unknown': r'Unknown'
            }
            
            for key, pattern in location_patterns.items():
                res = extract_row_data(loc_sec, pattern)
                data[f'{key}_N'], data[f'{key}_Pct'], data[f'{key}_Tot'], data[f'{key}_ERW'], data[f'{key}_Math'] = res

    except Exception as e:
        print(f"Error in {state_name}: {e}")
            
    return data

# --- Main Execution ---
print("Executing robust extraction...")
for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data)
df.to_csv('sat_final_ridge_data.csv', index=False)
print("\nDone! Verified 'Inc_Lowest' and 'Inc_Unknown' logic applied.")

Executing robust extraction...

Done! Verified 'Inc_Lowest' and 'Inc_Unknown' logic applied.


In [30]:
import pdfplumber
import pandas as pd
import os
import re

pdf_dir = './state_pdfs'
all_state_data = []

def parse_row(line):
    """
    Extracts [Count, Percent, Total, ERW, Math] from a line.
    Uses the '%' sign as the anchor.
    """
    # Clean commas and normalize spaces
    line = re.sub(r'\s+', ' ', line).replace(',', '')
    parts = line.split(' ')
    
    # Find the index of the percentage part (e.g., '15%')
    for i, part in enumerate(parts):
        if '%' in part:
            try:
                # Count is to the left of %, Scores are to the right
                count = float(parts[i-1])
                pct = float(part.replace('%', '')) / 100.0
                total = float(parts[i+1])
                erw = float(parts[i+2])
                math = float(parts[i+3])
                return count, pct, total, erw, math
            except (ValueError, IndexError):
                continue
    return None

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    data = {'State': state_name}
    
    # Define our target rows
    income_rows = {
        '0 - 59,669': 'Inc_Lowest',
        '59,670 - 76,797': 'Inc_2ndLow',
        '76,798 - 95,223': 'Inc_Middle',
        '95,224 - 124,913': 'Inc_2ndHigh',
        '124,914': 'Inc_Highest',
        'No Response': 'Inc_Unknown'
    }

    try:
        with pdfplumber.open(pdf_path) as pdf:
            # 1. Participation (Page 4)
            p4_text = pdf.pages[3].extract_text()
            takers_m = re.search(r'SAT\s+Takers\D+([\d,]+)', p4_text or "", re.IGNORECASE)
            data['SAT_Takers'] = int(takers_m.group(1).replace(',', '')) if takers_m else 0

            # 2. Sequential Scanning (Pages 5-8)
            current_section = None
            for page in pdf.pages[4:8]:
                text = page.extract_text()
                if not text: continue
                
                for line in text.split('\n'):
                    # Section Switching Logic
                    if 'Family Income' in line:
                        current_section = 'INCOME'
                        continue
                    elif 'School Location' in line:
                        current_section = 'LOCATION'
                        continue
                    elif 'Gender' in line or 'Language' in line:
                        current_section = 'OTHER'
                    
                    # Data Extraction based on active section
                    if current_section == 'INCOME':
                        for label, key in income_rows.items():
                            if label in line:
                                stats = parse_row(line)
                                if stats:
                                    data[f'{key}_N'], data[f'{key}_Pct'], data[f'{key}_Tot'], \
                                    data[f'{key}_ERW'], data[f'{key}_Math'] = stats

                    elif current_section == 'LOCATION':
                        for loc in ['City', 'Suburb', 'Town/Rural', 'Unknown']:
                            if loc in line:
                                stats = parse_row(line)
                                if stats:
                                    # Rename Location 'Unknown' to distinguish from Income 'Unknown'
                                    key = 'Loc_Unknown' if loc == 'Unknown' else loc
                                    data[f'{key}_N'], data[f'{key}_Pct'], data[f'{key}_Tot'], \
                                    data[f'{key}_ERW'], data[f'{key}_Math'] = stats

    except Exception as e:
        print(f"Error in {state_name}: {e}")
            
    return data

# --- Execution ---
print("Scanning PDFs with State Machine Logic...")
for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data)
# Fill NaNs with 0 in case some rows were missing in specific PDFs
df = df.fillna(0)
df.to_csv('sat_final_ridge_data.csv', index=False)
print("\nSuccess! CSV generated.")

Scanning PDFs with State Machine Logic...

Success! CSV generated.


In [31]:
import pdfplumber
import pandas as pd
import os
import re

pdf_dir = './state_pdfs'
all_state_data = []

def parse_row_stats(line):
    """
    Finds the % sign in a line and extracts the surrounding data:
    [Count] [Percent] [Total] [ERW] [Math]
    """
    # Remove commas and normalize spaces
    clean_line = re.sub(r'\s+', ' ', line).replace(',', '').strip()
    parts = clean_line.split(' ')
    
    for i, part in enumerate(parts):
        if '%' in part:
            try:
                # Layout: Count(i-1), Percent(i), Total(i+1), ERW(i+2), Math(i+3)
                count = float(parts[i-1])
                pct = float(part.replace('%', '')) / 100.0
                total = float(parts[i+1])
                erw = float(parts[i+2])
                math = float(parts[i+3])
                return count, pct, total, erw, math
            except (ValueError, IndexError):
                continue
    return None

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    # Extract state name from filename (e.g., '2025-alabama-sat...')
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    data = {'State': state_name}
    
    # Mapping labels to our CSV column prefixes
    race_map = {
        'American Indian': 'Race_AmInd', 'Asian': 'Race_Asian', 'Black': 'Race_Black',
        'Hispanic': 'Race_Hispanic', 'Native Hawaiian': 'Race_NatHaw', 'White': 'Race_White',
        'Two or More': 'Race_Multi', 'No Response': 'Race_NoResp'
    }
    edu_map = {
        'No High School Diploma': 'Edu_NoHS', 'High School Diploma': 'Edu_HS',
        'Associate Degree': 'Edu_Assoc', 'Bachelor\'s Degree': 'Edu_Bach',
        'Graduate Degree': 'Edu_Grad', 'No Response': 'Edu_NoResp'
    }
    inc_map = {
        '0 - 59,669': 'Inc_Lowest', '59,670 - 76,797': 'Inc_2ndLow',
        '76,798 - 95,223': 'Inc_Middle', '95,224 - 124,913': 'Inc_2ndHigh',
        '124,914': 'Inc_Highest', 'No Response': 'Inc_NoResp'
    }
    loc_map = {
        'City': 'Loc_City', 'Suburb': 'Loc_Suburb', 
        'Town/Rural': 'Loc_TownRural', 'Unknown': 'Loc_Unknown'
    }

    try:
        with pdfplumber.open(pdf_path) as pdf:
            # --- 1. Page 4: Participation & Graduates ---
            p4_text = pdf.pages[3].extract_text() or ""
            takers_m = re.search(r'SAT\s+Takers\D+([\d,]+)', p4_text, re.IGNORECASE)
            data['SAT_Takers_Total'] = int(takers_m.group(1).replace(',', '')) if takers_m else 0
            
            grads_m = re.search(r'High\s+School\s+Graduates\D+([\d,]+)', p4_text, re.IGNORECASE)
            data['HS_Graduates_Total'] = int(grads_m.group(1).replace(',', '')) if grads_m else 0
            
            part_m = re.search(r'Participation\s+Rate\s+(\d+%)', p4_text, re.IGNORECASE)
            data['SAT_Participation_Rate'] = float(part_m.group(1).replace('%',''))/100 if part_m else 0.0

            # --- 2. Sequential Scan: Performance, Race, Edu, Income, Location ---
            current_section = None
            for page in pdf.pages[4:9]: # Scan pages 5 through 9
                text = page.extract_text()
                if not text: continue
                
                for line in text.split('\n'):
                    # Section Detectors
                    if 'Race / Ethnicity' in line: current_section = 'RACE'
                    elif 'Highest Level of Parental Education' in line: current_section = 'EDU'
                    elif 'Family Income' in line: current_section = 'INCOME'
                    elif 'School Location' in line: current_section = 'LOCATION'
                    elif 'Total' in line and 'Mean Score' in line:
                        # Grab the very first 'Total' row for state-wide averages
                        stats = parse_row_stats(line)
                        if stats and 'Mean_Total' not in data:
                            data['Mean_Total'], _, data['Mean_Total_Score'], data['Mean_ERW'], data['Mean_Math'] = stats
                    
                    # Data Extraction
                    if current_section == 'RACE':
                        for label, prefix in race_map.items():
                            if label in line:
                                stats = parse_row_stats(line)
                                if stats:
                                    data[f'{prefix}_N'], data[f'{prefix}_Pct'], data[f'{prefix}_Total'], \
                                    data[f'{prefix}_ERW'], data[f'{prefix}_Math'] = stats

                    elif current_section == 'EDU':
                        for label, prefix in edu_map.items():
                            if label in line:
                                stats = parse_row_stats(line)
                                if stats:
                                    data[f'{prefix}_N'], data[f'{prefix}_Pct'], data[f'{prefix}_Total'], \
                                    data[f'{prefix}_ERW'], data[f'{prefix}_Math'] = stats

                    elif current_section == 'INCOME':
                        for label, prefix in inc_map.items():
                            if label in line:
                                stats = parse_row_stats(line)
                                if stats:
                                    data[f'{prefix}_N'], data[f'{prefix}_Pct'], data[f'{prefix}_Total'], \
                                    data[f'{prefix}_ERW'], data[f'{prefix}_Math'] = stats

                    elif current_section == 'LOCATION':
                        for label, prefix in loc_map.items():
                            if label in line:
                                stats = parse_row_stats(line)
                                if stats:
                                    data[f'{prefix}_N'], data[f'{prefix}_Pct'], data[f'{prefix}_Total'], \
                                    data[f'{prefix}_ERW'], data[f'{prefix}_Math'] = stats

    except Exception as e:
        print(f"Error processing {state_name}: {e}")
            
    return data

# --- Main Logic ---
print("Starting Full Dataset Extraction...")
for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

# Final DataFrame Processing
df = pd.DataFrame(all_state_data)
df = df.fillna(0) # Ensure no blanks for the regression model
df.to_csv('sat_final_modeling_master_2025.csv', index=False)

print(f"\nSuccess! Processed {len(df)} states.")
print("File saved as: sat_final_modeling_master_2025.csv")

Starting Full Dataset Extraction...

Success! Processed 5 states.
File saved as: sat_final_modeling_master_2025.csv


In [32]:
import pdfplumber
import pandas as pd
import os
import re

pdf_dir = './state_pdfs'
all_state_data = []

def parse_full_row(line):
    """
    Extracts all 8 data points from a College Board row:
    [Number, Pct, Total_Mean, ERW_Mean, Math_Mean, Met_Both_%, Met_ERW_%, Met_Math_%]
    """
    # Clean commas and normalize spaces
    clean_line = re.sub(r'\s+', ' ', line).replace(',', '').strip()
    parts = clean_line.split(' ')
    
    # We find the first percentage (Pct of Takers) to anchor the data
    for i, part in enumerate(parts):
        if '%' in part:
            try:
                # Based on CB Table Layout:
                # Label... [Number] [Pct%] [Mean] [ERW] [Math] [Both%] [ERW%] [Math%]
                n_takers = float(parts[i-1])
                pct_takers = float(part.replace('%', '')) / 100.0
                mean_tot = float(parts[i+1])
                mean_erw = float(parts[i+2])
                mean_math = float(parts[i+3])
                
                # Benchmarks are the last three columns
                met_both = float(parts[i+4].replace('%', '')) / 100.0
                met_erw = float(parts[i+5].replace('%', '')) / 100.0
                met_math = float(parts[i+6].replace('%', '')) / 100.0
                
                return n_takers, pct_takers, mean_tot, mean_erw, mean_math, met_both, met_erw, met_math
            except (ValueError, IndexError):
                continue
    return None

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    state_name = file_name.replace('2025-', '').split('-')[0].title()
    data = {'State': state_name}
    
    # Mapping definitions
    sections = {
        'Race / Ethnicity': {
            'American Indian': 'Race_AmInd', 'Asian': 'Race_Asian', 'Black': 'Race_Black',
            'Hispanic': 'Race_Hispanic', 'Native Hawaiian': 'Race_NatHaw', 'White': 'Race_White',
            'Two or More': 'Race_Multi', 'No Response': 'Race_NoResp'
        },
        'Gender': {
            'Female': 'Gen_Female', 'Male': 'Gen_Male', 'No Response': 'Gen_NoResp'
        },
        'Highest Level of Parental Education': {
            'No High School Diploma': 'Edu_NoHS', 'High School Diploma': 'Edu_HS',
            'Associate Degree': 'Edu_Assoc', 'Bachelor\'s Degree': 'Edu_Bach',
            'Graduate Degree': 'Edu_Grad', 'No Response': 'Edu_NoResp'
        },
        'Family Income': {
            '0 - 59,669': 'Inc_Lowest', '59,670 - 76,797': 'Inc_2ndLow',
            '76,798 - 95,223': 'Inc_Middle', '95,224 - 124,913': 'Inc_2ndHigh',
            '124,914': 'Inc_Highest', 'No Response': 'Inc_NoResp'
        },
        'School Location': {
            'City': 'Loc_City', 'Suburb': 'Loc_Suburb', 
            'Town/Rural': 'Loc_TownRural', 'Unknown': 'Loc_Unknown'
        }
    }

    try:
        with pdfplumber.open(pdf_path) as pdf:
            # 1. Page 4: Global Participation
            p4_text = pdf.pages[3].extract_text() or ""
            takers_m = re.search(r'SAT\s+Takers\D+([\d,]+)', p4_text, re.IGNORECASE)
            data['Total_SAT_Takers'] = int(takers_m.group(1).replace(',', '')) if takers_m else 0
            
            part_m = re.search(r'Participation\s+Rate\s+(\d+%)', p4_text, re.IGNORECASE)
            data['Total_Participation_Rate'] = float(part_m.group(1).replace('%',''))/100 if part_m else 0.0

            # 2. Sequential Scanning for Detailed Tables
            current_sec_key = None
            for page in pdf.pages[4:9]:
                text = page.extract_text()
                if not text: continue
                
                for line in text.split('\n'):
                    # Check if line triggers a section change
                    for sec_header in sections.keys():
                        if sec_header in line:
                            current_sec_key = sec_header
                            break
                    
                    if current_sec_key:
                        for label, prefix in sections[current_sec_key].items():
                            if label in line:
                                stats = parse_full_row(line)
                                if stats:
                                    # Assigning the 8 extracted values
                                    (data[f'{prefix}_N'], data[f'{prefix}_Pct'], 
                                     data[f'{prefix}_Mean_Tot'], data[f'{prefix}_Mean_ERW'], data[f'{prefix}_Mean_Math'],
                                     data[f'{prefix}_Met_Both'], data[f'{prefix}_Met_ERW'], data[f'{prefix}_Met_Math']) = stats

    except Exception as e:
        print(f"Error in {state_name}: {e}")
            
    return data

# --- Execution ---
print("Extracting full feature set for Ridge/Random Forest modeling...")
for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data).fillna(0)
df.to_csv('sat_comprehensive_capstone_data.csv', index=False)
print(f"Success! Master file created with {len(df.columns)} features.")

Extracting full feature set for Ridge/Random Forest modeling...
Success! Master file created with 179 features.
